# Figure reproduction

This notebook contains the plotting/chart code separated from the main
computational analysis. It expects the corresponding result CSV files to exist
in `results/`.

The code was transferred from the development notebook and personal absolute
paths were replaced by repository-relative paths. Review output filenames
against the final manuscript figure numbering before the Zenodo release.


## Reference storm-event signals


In [ ]:
# Source development-notebook cell 9
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

parameters = ["Turbidity", "EC", "DO", "Temp"]
storm_ids  = [1, 2, 3, 4]

param_colors = {
    "Turbidity": "#2166ac",
    "EC":        "#4dac26",
    "DO":        "#d01c8b",
    "Temp":      "#f4a582",
}

param_units = {
    "Turbidity": "NTU",
    "EC":        "µS/cm",
    "DO":        "mg/L",
    "Temp":      "°C",
}

n_storms = len(storm_ids)
n_params = len(parameters)

fig = plt.figure(figsize=(5 * n_storms, 3 * n_params))
fig.suptitle("Reference Signals: All Storms × All Parameters",
             fontsize=14, fontweight="bold", y=0.92)

gs = gridspec.GridSpec(n_params, n_storms, figure=fig,
                       hspace=0.45, wspace=0.35)

for row, parameter in enumerate(parameters):
    for col, storm_id in enumerate(storm_ids):
        ax = fig.add_subplot(gs[row, col])

        signal = get_reference_series(
            all_storms=all_storms,
            storm_id=storm_id,
            parameter=parameter,
        )

        # Fix: extract values to drop the datetime index
        values = signal.values if hasattr(signal, "values") else np.asarray(signal)

        # Flag degenerate signals instead of crashing
        if np.nanstd(values) == 0:
            ax.text(0.5, 0.5, "constant/n(all zeros?)",
                    ha="center", va="center", transform=ax.transAxes,
                    fontsize=9, color="red")
            ax.set_title(f"Storm {storm_id}", fontsize=10, pad=4)
            if col == 0:
                ax.set_ylabel(f"{parameter}/n({param_units[parameter]})", fontsize=9)
            ax.spines[["top", "right"]].set_visible(False)
            continue

        ax.plot(values, color=param_colors[parameter], linewidth=1.2, alpha=0.9)

        # Mark event extremum
        #if parameter in ("Turbidity", "EC"):
            #ext_idx = np.nanargmax(values)
        #else:
            #ext_idx = np.nanargmin(values)
        #ax.axvline(ext_idx, color="black", linewidth=0.8, linestyle="--", alpha=0.5)
        #ax.scatter([ext_idx], [values[ext_idx]], color="black", s=25, zorder=5)

        ax.set_title(f"Storm {storm_id}", fontsize=10, pad=4)
        if col == 0:
            ax.set_ylabel(f"{parameter}/n({param_units[parameter]})", fontsize=9)
        if row == n_params - 1:
            ax.set_xlabel("Time step", fontsize=9)

        ax.tick_params(labelsize=8)
        ax.spines[["top", "right"]].set_visible(False)

plt.savefig("reference_signals.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
# Source development-notebook cell 10
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
from pathlib import Path

# ----------------------------------------------------
# Settings
# ----------------------------------------------------
parameters = ["Turbidity", "EC", "DO", "Temp"]
storm_ids  = [1, 2, 3, 4]

param_colors = {
    "Turbidity": "#2166ac",
    "EC":        "#4dac26",
    "DO":        "#d01c8b",
    "Temp":      "#f4a582",
}

param_units = {
    "Turbidity": "NTU",
    "EC":        "µS/cm",
    "DO":        "%",
    "Temp":      "°C",
}

param_labels = {
    "Turbidity": "Turbidity",
    "EC":        "Electrical conductivity",
    "DO":        "Dissolved oxygen",
    "Temp":      "Water temperature",
}

output_dir = Path("figure_outputs")
output_dir.mkdir(exist_ok=True)

# ----------------------------------------------------
# Figure style
# ----------------------------------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 12,
    "axes.titlesize": 12,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.titlesize": 15,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white"
})

# ----------------------------------------------------
# Helper functions
# ----------------------------------------------------
def get_signal_values(all_storms, storm_id, parameter):
    """
    Return a 1D numpy array for a storm parameter.
    """

    signal = get_reference_series(
        all_storms=all_storms,
        storm_id=storm_id,
        parameter=parameter,
    )

    values = signal.values if hasattr(signal, "values") else np.asarray(signal)
    values = np.asarray(values, dtype=float)

    return values


def clean_invert_normalize_radar(
    all_storms,
    storm_id,
    radar_parameter="Radar",
    low_atol=1.0,
    high_threshold=5.0,
    interpolate=True
):
    """
    Clean, invert, and normalize radar-derived stage.

    Cleaning:
    - Converts to numeric
    - Removes near-zero artifacts
    - Removes unrealistically high values
    - Optionally interpolates short gaps

    Inversion:
    - Uses max(cleaned radar) - radar so storm response rises upward

    Normalization:
    - Scales from 0 to 1 for plotting on secondary axis
    """

    radar = get_reference_series(
        all_storms=all_storms,
        storm_id=storm_id,
        parameter=radar_parameter,
    )

    radar = pd.Series(radar).copy()
    radar = pd.to_numeric(radar, errors="coerce")

    # Remove radar artifacts
    radar[(np.isclose(radar, 0, atol=low_atol)) | (radar > high_threshold)] = np.nan

    # Optional: interpolate gaps caused by cleaning
    if interpolate:
        radar = radar.interpolate(limit_direction="both")

    values = radar.to_numpy(dtype=float)

    finite = values[np.isfinite(values)]

    if len(finite) == 0:
        return values

    # Invert radar so larger plotted values indicate higher stage response
    radar_inv = np.nanmax(finite) - values

    # Normalize to 0–1 for secondary axis
    finite_inv = radar_inv[np.isfinite(radar_inv)]

    if len(finite_inv) == 0 or np.nanmax(finite_inv) == np.nanmin(finite_inv):
        return radar_inv

    radar_norm = (
        (radar_inv - np.nanmin(finite_inv)) /
        (np.nanmax(finite_inv) - np.nanmin(finite_inv))
    )

    return radar_norm


def robust_ylim(values, pad_fraction=0.08):
    """
    Create y-limits using finite values and padding.
    """

    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return None

    ymin = np.nanmin(values)
    ymax = np.nanmax(values)

    yrange = ymax - ymin

    if yrange == 0:
        pad = 1
    else:
        pad = pad_fraction * yrange

    return ymin - pad, ymax + pad


In [ ]:
# Source development-notebook cell 11
# ----------------------------------------------------
# Create figure
# ----------------------------------------------------
n_storms = len(storm_ids)
n_params = len(parameters)

fig = plt.figure(figsize=(14, 8.8)) #4.8 * n_storms, 2.8 * n_params

gs = gridspec.GridSpec(
    n_params,
    n_storms,
    figure=fig,
    hspace=0.18, #0.32
    wspace=0.16 #0.28
)

plt.subplots_adjust(
    left=0.07,
    right=0.97,
    top=0.90,
    bottom=0.08
)

for row, parameter in enumerate(parameters):

    for col, storm_id in enumerate(storm_ids):

        ax = fig.add_subplot(gs[row, col])

        values = get_signal_values(
            all_storms=all_storms,
            storm_id=storm_id,
            parameter=parameter
        )

        x = np.arange(len(values))

        ax.plot(
            x,
            values,
            color=param_colors[parameter],
            linewidth=1.5,
            alpha=0.95
        )

        # ------------------------------------------------
        # Add cleaned, inverted, normalized radar only to turbidity row
        # ------------------------------------------------
        if parameter == "Turbidity":

            radar_norm = clean_invert_normalize_radar(
                all_storms=all_storms,
                storm_id=storm_id,
                radar_parameter="Radar",
                low_atol=1.0,
                high_threshold=5.0,
                interpolate=True
            )

            ax2 = ax.twinx()
            
            radar_color = "red"    
            ax2.plot(
                np.arange(len(radar_norm)),
                radar_norm,
                color=radar_color,
                linewidth=1.2,
                linestyle="--",
                alpha=0.65
            )

            ax2.set_ylim(-0.05, 1.2)

            if col == n_storms - 1:
                ax2.set_ylabel(
                    "Water level (m)",
                    fontsize=13,
                    fontweight="normal",
                    color=radar_color
                )
                ax2.tick_params(axis="y", labelsize=9, colors=radar_color)
            else:
                ax2.set_yticklabels([])
                ax2.tick_params(axis="y", length=0)

            ax2.spines["top"].set_visible(False)

        # Titles only in first row
        if row == 0:
            ax.set_title(
                f"Event {storm_id}",
                fontsize=12,
                fontweight="bold",
                pad=6
            )

        # Y-label only first column
        if col == 0:
            ax.set_ylabel(
                f"{param_labels[parameter]}/n({param_units[parameter]})",
                fontsize=13,
                fontweight="bold"
            )

        # X-label only bottom row
        if row == n_params - 1:
            ax.set_xlabel("Time step", fontsize=16)
        else:
            ax.set_xticklabels([])

        # Adaptive y-limit per subplot for readability
        ylim = robust_ylim(values)
        if ylim is not None:
            ax.set_ylim(ylim)

        ax.tick_params(axis="both", labelsize=13, length=3)
        ax.locator_params(axis="x", nbins=4)
        ax.locator_params(axis="y", nbins=4)

        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(True, alpha=0.20, linewidth=0.6)

#fig.suptitle(
    #"Reference Water-Quality Signals for Selected Storm Events",
    #fontsize=15,
    #fontweight="bold",
    #y=0.995
#)

#fig.tight_layout(rect=[0, 0, 1, 0.975])

png_path = output_dir / "Figure_1_reference_signals_radar_secondary_adaptive_ylim.png"
pdf_path = output_dir / "Figure_1_reference_signals_radar_secondary_adaptive_ylim.pdf"
svg_path = output_dir / "Figure_1_reference_signals_radar_secondary_adaptive_ylim.svg"

fig.savefig(png_path, dpi=600, bbox_inches="tight")
fig.savefig(pdf_path, bbox_inches="tight")
fig.savefig(svg_path, bbox_inches="tight")

plt.show()

print("Saved:")
print(png_path)
print(pdf_path)
print(svg_path)


## Algorithm ranking and error summaries


In [ ]:
# Source development-notebook cell 29
def savefig(name):
    plt.tight_layout()
    plt.savefig(figdir / name, dpi=300, bbox_inches="tight")
    plt.show()


In [ ]:
# Source development-notebook cell 30
for scenario in overall_by_scenario["scenario"].unique():
    df_plot = overall_by_scenario[
        overall_by_scenario["scenario"] == scenario
    ].copy()

    df_plot = df_plot.sort_values("Composite_Rank", ascending=True)

    plt.figure(figsize=(9, 5))
    plt.barh(df_plot["algorithm"], df_plot["Composite_Rank"])
    plt.gca().invert_yaxis()
    plt.xlabel("Mean composite rank")#lower is better
    plt.ylabel("Algorithm")
    plt.title(f"Overall Fixed-Rule Algorithm Ranking: {scenario}")
    savefig(f"overall_rank_{scenario}.png")


In [ ]:
# Source development-notebook cell 31
import matplotlib as mpl

mpl.rcParams.update({
    "font.family"       : "serif",
    "font.serif"        : ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset"  : "stix",        # STIX matches Times for math symbols
    "axes.titlesize"    : 11,
    "axes.labelsize"    : 11,
    "xtick.labelsize"   : 11,
    "ytick.labelsize"   : 11,
    "figure.dpi"        : 300,
})


In [ ]:
# Source development-notebook cell 32
rmse_scenario = (
    summary
    .groupby(["scenario", "algorithm"], as_index=False)["RMSE_mean"]
    .mean()
)

for scenario in rmse_scenario["scenario"].unique():
    df_plot = rmse_scenario[rmse_scenario["scenario"] == scenario].copy()
    df_plot = df_plot.sort_values("RMSE_mean", ascending=True)

    plt.figure(figsize=(9, 5))
    plt.barh(df_plot["algorithm"], df_plot["RMSE_mean"])
    plt.gca().invert_yaxis()
    plt.xlabel("Mean RMSE")
    plt.ylabel("Algorithm")
    plt.title(f"Mean RMSE by Algorithm: {scenario}")
    savefig(f"rmse_by_algorithm_{scenario}.png")


In [ ]:
# Source development-notebook cell 33
mae_scenario = (
    summary
    .groupby(["scenario", "algorithm"], as_index=False)["MAE_mean"]
    .mean()
)

for scenario in mae_scenario["scenario"].unique():
    df_plot = mae_scenario[mae_scenario["scenario"] == scenario].copy()
    df_plot = df_plot.sort_values("MAE_mean", ascending=True)

    plt.figure(figsize=(9, 5))
    plt.barh(df_plot["algorithm"], df_plot["MAE_mean"])
    plt.gca().invert_yaxis()
    plt.xlabel("Mean MAE")
    plt.ylabel("Algorithm")
    plt.title(f"Mean MAE by Algorithm: {scenario}")
    savefig(f"mae_by_algorithm_{scenario}.png")


In [ ]:
# Source development-notebook cell 34
for parameter in parameter_rank["parameter"].unique():
    df_plot = parameter_rank[
        parameter_rank["parameter"] == parameter
    ].copy()

    df_plot = df_plot.sort_values("Composite_Rank", ascending=True)

    plt.figure(figsize=(9, 5))
    plt.barh(df_plot["algorithm"], df_plot["Composite_Rank"])
    plt.gca().invert_yaxis()
    plt.xlabel("Mean composite rank, lower is better")
    plt.ylabel("Algorithm")
    plt.title(f"Parameter-Specific Algorithm Ranking: {parameter}")
    savefig(f"parameter_rank_{parameter}.png")


In [ ]:
# Source development-notebook cell 35
for scenario in summary["scenario"].unique():
    df_s = summary[summary["scenario"] == scenario].copy()

    pivot = df_s.pivot_table(
        index="algorithm",
        columns="parameter",
        values="RMSE_mean",
        aggfunc="mean"
    )

    pivot = pivot.loc[pivot.mean(axis=1).sort_values().index]

    plt.figure(figsize=(8, max(4, 0.35 * len(pivot))))
    plt.imshow(pivot.values, aspect="auto")
    plt.colorbar(label="Mean RMSE")
    plt.xticks(range(len(pivot.columns)), pivot.columns, rotation=45, ha="right")
    plt.yticks(range(len(pivot.index)), pivot.index)
    plt.title(f"Mean RMSE Heatmap: {scenario}")
    savefig(f"rmse_heatmap_{scenario}.png")


In [ ]:
# Source development-notebook cell 42
# ============================================================
# Figure: Overall algorithm rank by parameter
# ============================================================

for parameter in parameters:
    df_plot = parameter_overall_rank[
        parameter_overall_rank["parameter"] == parameter
    ].copy()

    df_plot = df_plot.sort_values("Composite_Rank", ascending=True)

    plt.figure(figsize=(10, 6), facecolor="white")
    ax = plt.gca()
    ax.set_facecolor("white")

    ax.barh(df_plot["algorithm"], df_plot["Composite_Rank"])
    ax.invert_yaxis()

    ax.set_xlabel("Mean composite rank, lower is better")
    ax.set_ylabel("Algorithm")
    ax.set_title(f"Overall Fixed-Rule Algorithm Ranking: {parameter}")

    ax.grid(axis="x", linestyle="--", alpha=0.3)

    savefig_param(f"overall_algorithm_rank_{parameter}.png")


In [ ]:
# Source development-notebook cell 43
# ============================================================
# Figure: Algorithm ranking by parameter × scenario
# ============================================================

for parameter in parameters:
    for scenario in scenarios:
        df_plot = parameter_scenario_rank[
            (parameter_scenario_rank["parameter"] == parameter) &
            (parameter_scenario_rank["scenario"] == scenario)
        ].copy()

        if df_plot.empty:
            continue

        df_plot = df_plot.sort_values("Composite_Rank", ascending=True)

        plt.figure(figsize=(10, 6), facecolor="white")
        ax = plt.gca()
        ax.set_facecolor("white")

        ax.barh(df_plot["algorithm"], df_plot["Composite_Rank"])
        ax.invert_yaxis()

        ax.set_xlabel("Mean composite rank, lower is better")
        ax.set_ylabel("Algorithm")
        ax.set_title(f"{parameter}: Algorithm Ranking under {scenario}")

        ax.grid(axis="x", linestyle="--", alpha=0.3)

        savefig_param(f"rank_{parameter}_{scenario}.png")


In [ ]:
# Source development-notebook cell 44
# ============================================================
# Figure: Mean MAE by algorithm, parameter × scenario
# ============================================================

for parameter in parameters:
    for scenario in scenarios:
        df_plot = mae_by_param_scenario[
            (mae_by_param_scenario["parameter"] == parameter) &
            (mae_by_param_scenario["scenario"] == scenario)
        ].copy()

        if df_plot.empty:
            continue

        df_plot = df_plot.sort_values("MAE_mean", ascending=True)

        plt.figure(figsize=(10, 6), facecolor="white")
        ax = plt.gca()
        ax.set_facecolor("white")

        ax.barh(df_plot["algorithm"], df_plot["MAE_mean"])
        ax.invert_yaxis()

        ax.set_xlabel("Mean MAE")
        ax.set_ylabel("Algorithm")
        ax.set_title(f"{parameter}: Mean MAE by Algorithm under {scenario}")

        ax.grid(axis="x", linestyle="--", alpha=0.3)

        savefig_param(f"mae_{parameter}_{scenario}.png")


In [ ]:
# Source development-notebook cell 45
# ============================================================
# Figure: Mean MAE by algorithm, parameter × scenario
# ============================================================

for parameter in parameters:
    for scenario in scenarios:
        df_plot = mae_by_param_scenario[
            (mae_by_param_scenario["parameter"] == parameter) &
            (mae_by_param_scenario["scenario"] == scenario)
        ].copy()

        if df_plot.empty:
            continue

        df_plot = df_plot.sort_values("MAE_mean", ascending=True)

        plt.figure(figsize=(10, 6), facecolor="white")
        ax = plt.gca()
        ax.set_facecolor("white")

        ax.barh(df_plot["algorithm"], df_plot["MAE_mean"])
        ax.invert_yaxis()

        ax.set_xlabel("Mean MAE")
        ax.set_ylabel("Algorithm")
        ax.set_title(f"{parameter}: Mean MAE by Algorithm under {scenario}")

        ax.grid(axis="x", linestyle="--", alpha=0.3)

        savefig_param(f"mae_{parameter}_{scenario}.png")


In [ ]:
# Source development-notebook cell 46
# ============================================================
# Figure: Mean RMSE by algorithm, parameter × scenario
# ============================================================

for parameter in parameters:
    for scenario in scenarios:
        df_plot = rmse_by_param_scenario[
            (rmse_by_param_scenario["parameter"] == parameter) &
            (rmse_by_param_scenario["scenario"] == scenario)
        ].copy()

        if df_plot.empty:
            continue

        df_plot = df_plot.sort_values("RMSE_mean", ascending=True)

        plt.figure(figsize=(10, 6), facecolor="white")
        ax = plt.gca()
        ax.set_facecolor("white")

        ax.barh(df_plot["algorithm"], df_plot["RMSE_mean"])
        ax.invert_yaxis()

        ax.set_xlabel("Mean RMSE")
        ax.set_ylabel("Algorithm")
        ax.set_title(f"{parameter}: Mean RMSE by Algorithm under {scenario}")

        ax.grid(axis="x", linestyle="--", alpha=0.3)

        savefig_param(f"rmse_{parameter}_{scenario}.png")


In [ ]:
# Source development-notebook cell 47
# ============================================================
# Figure: RMSE heatmap by parameter and algorithm for each scenario
# ============================================================

for scenario in scenarios:
    df_s = summary[summary["scenario"] == scenario].copy()

    pivot = df_s.pivot_table(
        index="algorithm",
        columns="parameter",
        values="RMSE_mean",
        aggfunc="mean"
    )

    if pivot.empty:
        continue

    pivot = pivot.loc[pivot.mean(axis=1).sort_values().index]

    plt.figure(figsize=(8, max(4, 0.35 * len(pivot))), facecolor="white")
    ax = plt.gca()
    ax.set_facecolor("white")

    im = ax.imshow(pivot.values, aspect="auto")

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Mean RMSE")

    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=45, ha="right")

    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)

    ax.set_title(f"Mean RMSE by Algorithm and Parameter: {scenario}")

    savefig_param(f"rmse_heatmap_parameter_algorithm_{scenario}.png")


In [ ]:
# Source development-notebook cell 48
# ============================================================
# Figure: MAE heatmap by parameter and algorithm for each scenario
# ============================================================

for scenario in scenarios:
    df_s = summary[summary["scenario"] == scenario].copy()

    pivot = df_s.pivot_table(
        index="algorithm",
        columns="parameter",
        values="MAE_mean",
        aggfunc="mean"
    )

    if pivot.empty:
        continue

    pivot = pivot.loc[pivot.mean(axis=1).sort_values().index]

    plt.figure(figsize=(8, max(4, 0.35 * len(pivot))), facecolor="white")
    ax = plt.gca()
    ax.set_facecolor("white")

    im = ax.imshow(pivot.values, aspect="auto")

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Mean MAE")

    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=45, ha="right")

    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)

    ax.set_title(f"Mean MAE by Algorithm and Parameter: {scenario}")

    savefig_param(f"mae_heatmap_parameter_algorithm_{scenario}.png")


In [ ]:
# Source development-notebook cell 49
# ============================================================
# Figure: Shannon entropy difference by parameter × scenario
# ============================================================

if "Shannon_Entropy_Diff_mean" in summary.columns:
    entropy_by_param_scenario = (
        summary
        .groupby(["parameter", "scenario", "algorithm"], as_index=False)["Shannon_Entropy_Diff_mean"]
        .mean()
        .sort_values(["parameter", "scenario", "Shannon_Entropy_Diff_mean"])
    )

    entropy_by_param_scenario.to_csv(
        param_outdir / "entropy_diff_by_parameter_scenario_algorithm.csv",
        index=False
    )

    for parameter in parameters:
        for scenario in scenarios:
            df_plot = entropy_by_param_scenario[
                (entropy_by_param_scenario["parameter"] == parameter) &
                (entropy_by_param_scenario["scenario"] == scenario)
            ].copy()

            if df_plot.empty:
                continue

            df_plot = df_plot.sort_values("Shannon_Entropy_Diff_mean", ascending=True)

            plt.figure(figsize=(10, 6), facecolor="white")
            ax = plt.gca()
            ax.set_facecolor("white")

            ax.barh(df_plot["algorithm"], df_plot["Shannon_Entropy_Diff_mean"])
            ax.invert_yaxis()

            ax.set_xlabel("Mean Shannon entropy difference, lower is better")
            ax.set_ylabel("Algorithm")
            ax.set_title(f"{parameter}: Entropy Preservation under {scenario}")

            ax.grid(axis="x", linestyle="--", alpha=0.3)

            savefig_param(f"entropy_{parameter}_{scenario}.png")


In [ ]:
# Source development-notebook cell 50
# ============================================================
# Figure: Variance ratio by parameter × scenario
# ============================================================

if "Variance_Ratio_mean" in summary.columns:
    variance_by_param_scenario = (
        summary
        .groupby(["parameter", "scenario", "algorithm"], as_index=False)["Variance_Ratio_mean"]
        .mean()
        .sort_values(["parameter", "scenario", "Variance_Ratio_mean"])
    )

    variance_by_param_scenario.to_csv(
        param_outdir / "variance_ratio_by_parameter_scenario_algorithm.csv",
        index=False
    )

    for parameter in parameters:
        for scenario in scenarios:
            df_plot = variance_by_param_scenario[
                (variance_by_param_scenario["parameter"] == parameter) &
                (variance_by_param_scenario["scenario"] == scenario)
            ].copy()

            if df_plot.empty:
                continue

            df_plot = df_plot.sort_values("Variance_Ratio_mean", ascending=True)

            plt.figure(figsize=(10, 6), facecolor="white")
            ax = plt.gca()
            ax.set_facecolor("white")

            ax.barh(df_plot["algorithm"], df_plot["Variance_Ratio_mean"])
            ax.axvline(1.0, linestyle="--", linewidth=1)

            ax.set_xlabel("Mean variance ratio, ideal ≈ 1")
            ax.set_ylabel("Algorithm")
            ax.set_title(f"{parameter}: Variance Preservation under {scenario}")

            ax.grid(axis="x", linestyle="--", alpha=0.3)

            savefig_param(f"variance_{parameter}_{scenario}.png")


In [ ]:
# Source development-notebook cell 51
# ============================================================
# Figure: Best algorithm frequency by parameter
# ============================================================

best_each_condition = (
    ranked
    .sort_values("Composite_Rank")
    .groupby(
        ["parameter", "storm_id", "scenario", "severity", "evaluation_domain"],
        as_index=False
    )
    .first()
)

best_counts_param = (
    best_each_condition
    .groupby(["parameter", "algorithm"])
    .size()
    .reset_index(name="count")
    .sort_values(["parameter", "count"], ascending=[True, False])
)

best_counts_param.to_csv(
    param_outdir / "best_algorithm_frequency_by_parameter.csv",
    index=False
)

for parameter in parameters:
    df_plot = best_counts_param[
        best_counts_param["parameter"] == parameter
    ].copy()

    if df_plot.empty:
        continue

    df_plot = df_plot.sort_values("count", ascending=True)

    plt.figure(figsize=(10, 6), facecolor="white")
    ax = plt.gca()
    ax.set_facecolor("white")

    ax.barh(df_plot["algorithm"], df_plot["count"])

    ax.set_xlabel("Number of conditions ranked best")
    ax.set_ylabel("Algorithm")
    ax.set_title(f"Best-Ranked Algorithm Frequency: {parameter}")

    ax.grid(axis="x", linestyle="--", alpha=0.3)

    savefig_param(f"best_algorithm_frequency_{parameter}.png")


In [ ]:
# Source development-notebook cell 53
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

figdir = Path("analysis_outputs/figures/grouped")
figdir.mkdir(parents=True, exist_ok=True)


def savefig_grouped(name):
    plt.tight_layout()
    plt.savefig(figdir / name, dpi=300, bbox_inches="tight", facecolor="white")
    plt.show()


def grouped_bar_by_parameter(
    summary,
    scenario,
    metric="MAE_mean",
    title_metric="Mean MAE",
    max_algorithms=None
):
    """
    Creates a grouped bar chart for one scenario.

    x-axis: algorithms
    grouped bars: water-quality parameters
    y-axis: selected metric
    """

    df_s = summary[summary["scenario"] == scenario].copy()

    if metric not in df_s.columns:
        print(f"{metric} not found in summary.")
        return

    # Average across storms and severity, but keep algorithm and parameter
    df_plot = (
        df_s
        .groupby(["algorithm", "parameter"], as_index=False)[metric]
        .mean()
    )

    # Order algorithms by overall mean metric
    alg_order = (
        df_plot
        .groupby("algorithm")[metric]
        .mean()
        .sort_values(ascending=True)
        .index
        .tolist()
    )

    if max_algorithms is not None:
        alg_order = alg_order[:max_algorithms]

    df_plot = df_plot[df_plot["algorithm"].isin(alg_order)]

    pivot = df_plot.pivot(
        index="algorithm",
        columns="parameter",
        values=metric
    ).loc[alg_order]

    algorithms = pivot.index.tolist()
    parameters = pivot.columns.tolist()

    x = np.arange(len(algorithms))
    width = 0.8 / len(parameters)

    plt.figure(figsize=(max(10, 0.8 * len(algorithms)), 6), facecolor="white")
    ax = plt.gca()
    ax.set_facecolor("white")

    for i, parameter in enumerate(parameters):
        offset = (i - (len(parameters) - 1) / 2) * width
        ax.bar(
            x + offset,
            pivot[parameter].values,
            width,
            label=parameter
        )

    ax.set_xticks(x)
    ax.set_xticklabels(algorithms, rotation=45, ha="right")

    ax.set_ylabel(title_metric)
    ax.set_xlabel("Algorithm")
    ax.set_title(f"{title_metric} by Algorithm and Parameter: {scenario}")
    ax.legend(title="Parameter")

    ax.grid(axis="y", linestyle="--", alpha=0.3)

    savefig_grouped(f"grouped_{metric}_{scenario}.png")


In [ ]:
# Source development-notebook cell 56
def grouped_rank_by_parameter(
    ranked,
    scenario,
    metric="Composite_Rank",
    title_metric="Mean composite rank",
    max_algorithms=None
):
    """
    Grouped bar chart using ranked dataframe.

    x-axis: algorithms
    grouped bars: parameters
    y-axis: composite rank; #lower is better
    """

    df_s = ranked[ranked["scenario"] == scenario].copy()

    if metric not in df_s.columns:
        print(f"{metric} not found in ranked.")
        return

    df_plot = (
        df_s
        .groupby(["algorithm", "parameter"], as_index=False)[metric]
        .mean()
    )

    alg_order = (
        df_plot
        .groupby("algorithm")[metric]
        .mean()
        .sort_values(ascending=True)
        .index
        .tolist()
    )

    if max_algorithms is not None:
        alg_order = alg_order[:max_algorithms]

    df_plot = df_plot[df_plot["algorithm"].isin(alg_order)]

    pivot = df_plot.pivot(
        index="algorithm",
        columns="parameter",
        values=metric
    ).loc[alg_order]

    algorithms = pivot.index.tolist()
    parameters = pivot.columns.tolist()

    x = np.arange(len(algorithms))
    width = 0.8 / len(parameters)

    plt.figure(figsize=(max(10, 0.8 * len(algorithms)), 6), facecolor="white")
    ax = plt.gca()
    ax.set_facecolor("white")

    for i, parameter in enumerate(parameters):
        offset = (i - (len(parameters) - 1) / 2) * width
        ax.bar(
            x + offset,
            pivot[parameter].values,
            width,
            label=parameter
        )

    ax.set_xticks(x)
    ax.set_xticklabels(algorithms, ha="center")

    ax.set_ylabel(f"{title_metric}")
    ax.set_xlabel("Algorithm")
    ax.set_title(f"{title_metric} by Algorithm and Parameter: {scenario}")
    ax.legend(title="Parameter")

    ax.grid(axis="y", linestyle="--", alpha=0.3)

    savefig_grouped(f"grouped_{metric}_{scenario}.png")


In [ ]:
# Source development-notebook cell 71
heatmap_data = metric_summary.set_index("algorithm")[rank_cols]

plt.figure(figsize=(8, max(4, 0.45 * len(heatmap_data))), facecolor="white")
ax = plt.gca()
ax.set_facecolor("white")

im = ax.imshow(heatmap_data.values, aspect="auto")

cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Metric-specific rank")

ax.set_xticks(range(len(rank_cols)))
ax.set_xticklabels(
    ["RMSE", "MAE", "KGE", "NSE", "Pearson r", "Peak Error", "Variance ratio", "Entropy diff"],
    rotation=45,
    ha="right"
)

ax.set_yticks(range(len(heatmap_data.index)))
ax.set_yticklabels(heatmap_data.index)

ax.set_title(f"Multi-metric algorithm ranking under {scenario_name}")

# Add rank values inside cells
for i in range(heatmap_data.shape[0]):
    for j in range(heatmap_data.shape[1]):
        ax.text(
            j,
            i,
            f"{heatmap_data.iloc[i, j]:.0f}",
            ha="center",
            va="center",
            fontsize=9
        )

plt.tight_layout()
plt.savefig(
    figdir / f"multimetric_rank_heatmap_{scenario_name}.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)
plt.show()


In [ ]:
# Source development-notebook cell 77
import matplotlib.pyplot as plt

figdir = outdir / "figures"
figdir.mkdir(exist_ok=True)

for parameter in metric_by_parameter["parameter"].unique():

    df_p = metric_by_parameter[
        metric_by_parameter["parameter"] == parameter
    ].copy()

    df_p = df_p.sort_values("Mean_Metric_Rank")

    heatmap_data = df_p.set_index("algorithm")[rank_cols]

    plt.figure(figsize=(9, max(4, 0.45 * len(heatmap_data))), facecolor="white")
    ax = plt.gca()
    ax.set_facecolor("white")

    im = ax.imshow(heatmap_data.values, aspect="auto")

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Metric-specific rank")

    ax.set_xticks(range(len(rank_cols)))
    ax.set_xticklabels(
        ["RMSE", "MAE", "KGE", "NSE", "Pearson r", "Peak", "VR", "SE diff"],
        rotation=45,
        ha="right"
    )

    ax.set_yticks(range(len(heatmap_data.index)))
    ax.set_yticklabels(heatmap_data.index)

    ax.set_title(f"Multi-metric _{scenario_name}: {parameter}")

    for i in range(heatmap_data.shape[0]):
        for j in range(heatmap_data.shape[1]):
            ax.text(
                j,
                i,
                f"{heatmap_data.iloc[i, j]:.0f}",
                ha="center",
                va="center",
                fontsize=8
            )

    plt.tight_layout()
    plt.savefig(
        figdir / f"multimetric_rank_heatmap_{scenario_name}_{parameter}.png",
        dpi=300,
        bbox_inches="tight",
        facecolor="white"
    )
    plt.show()


In [ ]:
# Source development-notebook cell 78
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

figdir = outdir / "figures"
figdir.mkdir(exist_ok=True)

parameters = list(metric_by_parameter["parameter"].unique())
x_labels   = ["RMSE","MAE", "KGE", "NSE", "Pearson r", "Peak", "VR", "SE diff"]

# ── compute global rank range for consistent colour scale ─────────
all_values = []
for parameter in parameters:
    df_p = metric_by_parameter[metric_by_parameter["parameter"] == parameter]
    all_values.append(df_p.set_index("algorithm")[rank_cols].values)
vmin = min(v.min() for v in all_values)
vmax = max(v.max() for v in all_values)

# ── derive figure height from the tallest panel (most algorithms) ─
max_algos = max(
    len(metric_by_parameter[metric_by_parameter["parameter"] == p])
    for p in parameters
)
row_h  = 0.38
col_w  = 0.72
n_cols = len(rank_cols)

panel_w = n_cols * col_w
panel_h = max_algos * row_h
cbar_w  = 0.25
fig_w   = 2 * panel_w + cbar_w + 1.6
fig_h   = 2 * panel_h + 1.2

# ── layout constants (figure-fraction) ───────────────────────────
cbar_right   = 0.905
left_margin  = 0.10
right_edge   = cbar_right - 0.02
top_margin   = 0.93
bot_margin   = 0.10

total_w      = right_edge - left_margin
col_gap      = 0.08
panel_w_frac = (total_w - col_gap) / 2

total_h      = top_margin - bot_margin
row_gap      = 0.08          # ← reduce this to close row spacing
panel_h_frac = (total_h - row_gap) / 2

def panel_rect(row, col):
    x = left_margin + col * (panel_w_frac + col_gap)
    y = top_margin - (row + 1) * panel_h_frac - row * row_gap
    return [x, y, panel_w_frac, panel_h_frac]

fig  = plt.figure(figsize=(fig_w, fig_h), facecolor="white")
axes = []
ims  = []

for idx, parameter in enumerate(parameters):
    row, col = divmod(idx, 2)
    ax = fig.add_axes(panel_rect(row, col))
    ax.set_facecolor("white")
    axes.append(ax)

    df_p = (
        metric_by_parameter[metric_by_parameter["parameter"] == parameter]
        .copy()
        .sort_values("Mean_Metric_Rank")
    )
    heatmap_data = df_p.set_index("algorithm")[rank_cols]
    n_alg = len(heatmap_data)

    im = ax.imshow(
        heatmap_data.values,
        aspect="auto",
        vmin=vmin,
        vmax=vmax,
        cmap="YlOrRd",
        #interpolation="bilinear"
    )
    ims.append(im)

    ax.set_xticks(range(n_cols))
    ax.set_xticklabels(x_labels, rotation=40, ha="right", fontsize=8)
    ax.tick_params(axis="x", pad=2)

    ax.set_yticks(range(n_alg))
    ax.set_yticklabels(heatmap_data.index, fontsize=8)
    ax.tick_params(axis="y", pad=2)

    ax.set_title(f"({chr(97+idx)}) {parameter}", fontsize=10, fontweight="bold", pad=5)

    threshold = vmin + (vmax - vmin) * 0.60
    for i in range(n_alg):
        for j in range(n_cols):
            val = heatmap_data.iloc[i, j]
            fc  = "white" if val > threshold else "black"
            ax.text(j, i, f"{val:.0f}",
                    ha="center", va="center",
                    fontsize=7.5, color=fc, fontweight="bold")

    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(length=0)

# ── shared colorbar ───────────────────────────────────────────────
cbar_ax = fig.add_axes([0.905, bot_margin, 0.018, top_margin - bot_margin])
cbar = fig.colorbar(ims[0], cax=cbar_ax)
cbar.set_label("Metric rank  (1 = best)", fontsize=9, labelpad=6)
cbar.ax.tick_params(labelsize=8)
cbar.outline.set_visible(False)

fig.suptitle(
    f"Multi-metric ranking heatmap — {scenario_name}",
    fontsize=11, fontweight="bold", y=0.975,
)

plt.savefig(
    figdir / f"multimetric_rank_heatmap_{scenario_name}_combined.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)
plt.show()


In [ ]:
# Source development-notebook cell 79
import matplotlib.pyplot as plt
import numpy as np

figdir = outdir / "figures"
figdir.mkdir(exist_ok=True)

parameter_order = ["Turbidity", "EC", "DO", "Temp"]
parameters = [p for p in parameter_order if p in metric_by_parameter["parameter"].unique()]

if len(parameters) == 0:
    parameters = list(metric_by_parameter["parameter"].unique())

metric_labels = ["RMSE", "MAE", "KGE", "NSE", "Pearson r", "Peak", "VR", "SE diff"]

# Store heatmap data
heatmap_store = {}
all_values = []

for parameter in parameters:
    df_p = metric_by_parameter[
        metric_by_parameter["parameter"] == parameter
    ].copy()

    df_p = df_p.sort_values("Mean_Metric_Rank")
    heatmap_data = df_p.set_index("algorithm")[rank_cols]

    heatmap_store[parameter] = heatmap_data
    all_values.extend(heatmap_data.values.flatten())

vmin = np.nanmin(all_values)
vmax = np.nanmax(all_values)

# Use constrained_layout instead of tight_layout
fig, axes = plt.subplots(
    2,
    2,
    figsize=(18, 12),
    facecolor="white",
    constrained_layout=True
)

axes = axes.flatten()

for ax, parameter in zip(axes, parameters):
    heatmap_data = heatmap_store[parameter]

    im = ax.imshow(
        heatmap_data.values,
        aspect="auto",
        vmin=vmin,
        vmax=vmax
    )

    ax.set_facecolor("white")

    ax.set_xticks(range(len(rank_cols)))
    ax.set_xticklabels(
        metric_labels,
        rotation=45,
        ha="right",
        fontsize=11
    )

    ax.set_yticks(range(len(heatmap_data.index)))
    ax.set_yticklabels(
        heatmap_data.index,
        fontsize=9
    )

    ax.tick_params(axis="y", pad=6)
    ax.tick_params(axis="x", pad=4)

    ax.set_title(
        parameter,
        fontsize=13,
        fontweight="bold",
        pad=10
    )

    for i in range(heatmap_data.shape[0]):
        for j in range(heatmap_data.shape[1]):
            ax.text(
                j,
                i,
                f"{heatmap_data.iloc[i, j]:.0f}",
                ha="center",
                va="center",
                fontsize=11
            )

for ax in axes[len(parameters):]:
    ax.axis("off")

# Put one shared colorbar outside the plots
cbar = fig.colorbar(
    im,
    ax=axes,
    location="right",
    shrink=0.85,
    pad=0.03
)

cbar.set_label(
    "Metric-specific rank",
    fontsize=11
)

fig.suptitle(
    f"Multi-metric {scenario_name.replace('_', ' ').title()} Ranking by Parameter",
    fontsize=16,
    fontweight="bold"
)

plt.savefig(
    figdir / f"multimetric_rank_heatmap_{scenario_name}_2x2_clean.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()


## Field-realistic fault reconstruction examples


In [ ]:
# Source development-notebook cell 81
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ------------------------------------------------------------
# User choices
# ------------------------------------------------------------
storm_id = "Storm_2"          # change to your storm ID
parameter = "Turbidity"       # e.g. "Turbidity", "EC", "DO", "Temperature"
scenario = "random_missing"   # or "field_realistic_fault", "random_missing", "block_missing"
severity = "severe"         # "moderate" or "severe"
rep_seed = 44                 # choose any integer for reproducibility

# Optional: algorithm subset. Set to None to plot all algorithms in pool
algorithm_subset = None
# Example:
# algorithm_subset = ["Wavelet", "CEEMDAN", "Median_Filter", "Savitzky_Golay"]

outdir = Path("analysis_outputs/sample_reconstructions")
outdir.mkdir(parents=True, exist_ok=True)


In [ ]:
# Source development-notebook cell 82
# ------------------------------------------------------------
# Get reference signal
# ------------------------------------------------------------
x_ref = get_reference_signal(all_storms, 2, parameter).astype(float)
t = np.arange(len(x_ref))

rng = np.random.default_rng(rep_seed)

# Apply degradation
x_deg, mask = DEGRADATION_FUNCTIONS[scenario](x_ref.copy(), severity, rng)

print("Signal length:", len(x_ref))
print("Scenario:", scenario)
print("Severity:", severity)
print("Parameter:", parameter)
print("Storm:", 2)
print("Number affected:", np.sum(mask) if mask is not None else "N/A")


In [ ]:
# Source development-notebook cell 83
# ------------------------------------------------------------
# Choose algorithms
# ------------------------------------------------------------
algorithm_pool = choose_algorithm_pool(scenario)

if algorithm_subset is not None:
    algorithm_pool = {k: v for k, v in algorithm_pool.items() if k in algorithm_subset}

print("Algorithms to plot:", list(algorithm_pool.keys()))

# ------------------------------------------------------------
# Run reconstruction / correction
# ------------------------------------------------------------
reconstructions = {}

for alg_name, alg_func in algorithm_pool.items():
    try:
        # Most of your functions likely use x_deg directly
        x_hat = alg_func(x_deg.copy())
        x_hat = np.asarray(x_hat, dtype=float)

        # Defensive truncation if needed
        n = min(len(x_ref), len(x_hat))
        x_hat = x_hat[:n]

        reconstructions[alg_name] = x_hat
    except Exception as e:
        print(f"{alg_name} failed: {e}")
        reconstructions[alg_name] = None


In [ ]:
# Source development-notebook cell 84
# ------------------------------------------------------------
# Plot helper
# ------------------------------------------------------------
valid_algs = [k for k, v in reconstructions.items() if v is not None]
n_algs = len(valid_algs)

ncols = 2
nrows = int(np.ceil((n_algs + 1) / ncols))   # +1 for top degraded panel

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(14, 3.5 * nrows),
    facecolor="white"
)

axes = np.atleast_1d(axes).flatten()

# ------------------------------------------------------------
# Panel 1: reference + degraded
# ------------------------------------------------------------
ax0 = axes[0]
ax0.plot(t, x_ref, label="Reference", linewidth=2)

if scenario in ["random_missing", "block_missing"]:
    ax0.plot(t, x_deg, label="Degraded", linewidth=1.5)
    # mark missing positions
    missing_idx = np.where(mask)[0]
    if len(missing_idx) > 0:
        ax0.scatter(
            missing_idx,
            x_ref[missing_idx],
            label="Missing locations",
            marker="x",
            s=25
        )
else:
    ax0.plot(t, x_deg, label="Degraded", linewidth=1.5)

ax0.set_title(f"Reference vs Degraded/n{parameter} | {storm_id} | {scenario} | {severity}")
ax0.set_xlabel("Time index")
ax0.set_ylabel(parameter)
ax0.legend()
ax0.grid(alpha=0.3)

# ------------------------------------------------------------
# Remaining panels: each algorithm
# ------------------------------------------------------------
for i, alg_name in enumerate(valid_algs, start=1):
    ax = axes[i]
    x_hat = reconstructions[alg_name]

    n = min(len(t), len(x_hat))
    ax.plot(t[:n], x_ref[:n], label="Reference", linewidth=2)
    ax.plot(t[:n], x_hat[:n], label=alg_name, linewidth=1.5)

    if scenario in ["random_missing", "block_missing"]:
        missing_idx = np.where(mask[:n])[0]
        if len(missing_idx) > 0:
            ax.scatter(
                missing_idx,
                x_ref[:n][missing_idx],
                label="Missing locations",
                marker="x",
                s=20
            )

    ax.set_title(alg_name)
    ax.set_xlabel("Time index")
    ax.set_ylabel(parameter)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

# Hide unused axes
for j in range(n_algs + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
save_path = outdir / f"sample_reconstruction_grid_{scenario}_{parameter}_{storm_id}_{severity}.png"
plt.savefig(save_path, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

print("Saved:", save_path)


In [ ]:
# Source development-notebook cell 85
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ------------------------------------------------------------
# User choices
# ------------------------------------------------------------
storm_num = 4
storm_label = f"Storm_{storm_num}"
parameter = "DO"  # e.g. "Turbidity", "EC", "DO", "Temp"
scenario = "field_realistic_fault"   # random_missing, block_missing, additive_noise, field_realistic_fault
severity = "severe"
rep_seed = 44

# None = plot every algorithm available for the scenario
algorithm_subset = None

outdir = Path("analysis_outputs/sample_reconstructions")
outdir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Reference signal
# ------------------------------------------------------------
x_ref = get_reference_signal(all_storms, storm_num, parameter).astype(float)
t = np.arange(len(x_ref))

rng = np.random.default_rng(rep_seed)
x_deg, mask = DEGRADATION_FUNCTIONS[scenario](x_ref.copy(), severity, rng)

print("Signal length:", len(x_ref))
print("Scenario:", scenario)
print("Severity:", severity)
print("Parameter:", parameter)
print("Storm:", storm_label)
print("Number affected:", np.sum(mask) if mask is not None else "N/A")

# ------------------------------------------------------------
# Helper: extract prediction from functions that may also
# return diagnostics
# ------------------------------------------------------------
def extract_prediction(result):
    return result[0] if isinstance(result, tuple) else result

# ------------------------------------------------------------
# Get hydrologic series aligned EXACTLY with x_ref
# Only needed for missingness scenarios
# ------------------------------------------------------------
def get_aligned_hydro(all_storms, storm_num, parameter, x_ref):
    df = get_storm_dataframe(all_storms, storm_num).copy()

    pcol = find_parameter_column(df, parameter)
    if pcol is None:
        raise ValueError(f"Could not identify {parameter} column.")

    wq = pd.to_numeric(df[pcol], errors="coerce")
    valid = wq.notna()
    ref_index = df.index[valid]
    wq_check = wq.loc[valid].to_numpy(dtype=float)

    if len(wq_check) != len(x_ref):
        raise ValueError(f"WQ alignment mismatch: {len(wq_check)} vs {len(x_ref)}.")

    if not np.allclose(wq_check, x_ref, equal_nan=True):
        raise ValueError("Aligned WQ series does not match x_ref.")

    # Prefer Stage_smoothed because this matches the original
    # hydrology-informed experiment
    if "classified_storms" in globals():
        try:
            hdf = get_storm_dataframe(classified_storms, storm_num).copy()

            if "Stage_smoothed" in hdf.columns:
                stage = pd.to_numeric(hdf["Stage_smoothed"], errors="coerce")
                stage = stage.reindex(ref_index).to_numpy(dtype=float)

                if len(stage) == len(x_ref) and np.isfinite(stage).mean() > 0.95:
                    return stage, "Stage_smoothed"
        except Exception:
            pass

    # Fallback: simultaneously measured Radar from same storm
    if "Radar" not in df.columns:
        raise ValueError("Neither aligned Stage_smoothed nor Radar is available.")

    radar = pd.to_numeric(df["Radar"], errors="coerce").loc[valid].to_numpy(dtype=float)

    if len(radar) != len(x_ref):
        raise ValueError(f"Radar alignment mismatch: {len(radar)} vs {len(x_ref)}.")

    return radar, "Radar"

# ------------------------------------------------------------
# Choose algorithms
# ------------------------------------------------------------
algorithm_pool = choose_algorithm_pool(scenario).copy()

if scenario in ["random_missing", "block_missing"]:
    hydro, hydro_source = get_aligned_hydro(all_storms, storm_num, parameter, x_ref)
    print("Hydrologic auxiliary:", hydro_source)
    print("Hydrologic length:", len(hydro))
    print("Finite hydrologic values:", f"{100*np.isfinite(hydro).mean():.1f}%")

    linear_key = next((k for k in ["Linear_Interpolation", "Linear Interpolation", "Linear"] if k in algorithm_pool), None)
    if linear_key is None:
        raise KeyError(f"Linear interpolation not found. Available: {list(algorithm_pool)}")

    linear_fn = algorithm_pool[linear_key]

    algorithm_pool["Local_Moving_Mean"] = lambda x: local_moving_mean_gapfill(x=x, n_each_side=2)
    algorithm_pool["Delta_H_Weighted_Interpolation"] = lambda x: delta_h_weighted_gapfill(x=x, stage=hydro, linear_function=linear_fn)
    algorithm_pool["HPWI"] = lambda x: hpwi_gapfill(x=x, stage=hydro, linear_function=linear_fn)

if algorithm_subset is not None:
    algorithm_pool = {k: v for k, v in algorithm_pool.items() if k in algorithm_subset}

print("Algorithms to plot:", list(algorithm_pool.keys()))

# ------------------------------------------------------------
# Run reconstruction / correction
# ------------------------------------------------------------
reconstructions = {}

for alg_name, alg_func in algorithm_pool.items():
    try:
        result = alg_func(x_deg.copy())
        x_hat = np.asarray(extract_prediction(result), dtype=float)
        x_hat = x_hat[:min(len(x_ref), len(x_hat))]
        reconstructions[alg_name] = x_hat
    except Exception as e:
        print(f"{alg_name} failed: {e}")
        reconstructions[alg_name] = None

# ------------------------------------------------------------
# Plot setup
# ------------------------------------------------------------
valid_algs = [k for k, v in reconstructions.items() if v is not None]
n_algs = len(valid_algs)

ncols = 3
nrows = int(np.ceil((n_algs + 1) / ncols))

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(14, 3.5*nrows), facecolor="white")
axes = np.atleast_1d(axes).flatten()

# ------------------------------------------------------------
# Panel 1: Reference + degraded
# ------------------------------------------------------------
ax0 = axes[0]
ax0.plot(t, x_ref, label="Reference", linewidth=2)
ax0.plot(t, x_deg, label="Degraded", linewidth=1.5)

if scenario in ["random_missing", "block_missing"]:
    missing_idx = np.where(mask)[0]
    if len(missing_idx):
        ax0.scatter(missing_idx, x_ref[missing_idx], label="Missing locations", marker="x", s=25)

ax0.set_title(f"Reference vs Degraded/n{parameter} | {storm_label} | {scenario} | {severity}")
ax0.set_xlabel("Time index")
ax0.set_ylabel(parameter)
ax0.legend()
ax0.grid(alpha=0.3)

# ------------------------------------------------------------
# Remaining panels: each algorithm
# ------------------------------------------------------------
for i, alg_name in enumerate(valid_algs, start=1):
    ax = axes[i]
    x_hat = reconstructions[alg_name]
    n = min(len(t), len(x_hat))

    ax.plot(t[:n], x_ref[:n], label="Reference", linewidth=2)
    ax.plot(t[:n], x_hat[:n], label=alg_name, linewidth=1.5)

    if scenario in ["random_missing", "block_missing"]:
        missing_idx = np.where(mask[:n])[0]
        if len(missing_idx):
            ax.scatter(missing_idx, x_ref[:n][missing_idx], label="Missing locations", marker="x", s=20)

    ax.set_title(alg_name)
    ax.set_xlabel("Time index")
    ax.set_ylabel(parameter)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

# Hide unused axes
for j in range(n_algs + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()

save_path = outdir / f"sample_reconstruction_grid_{scenario}_{parameter}_{storm_label}_{severity}.png"
plt.savefig(save_path, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

print("Saved:", save_path)


In [ ]:
# Source development-notebook cell 86
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ------------------------------------------------------------
# User choices
# ------------------------------------------------------------
storm_num = 4
storm_label = f"Storm_{storm_num}"
parameter = "DO"  # e.g. "Turbidity", "EC", "DO", "Temp"
scenario = "field_realistic_fault"   # random_missing, block_missing, additive_noise, field_realistic_fault
severity = "severe"
rep_seed = 44

# None = plot every algorithm available for the scenario
algorithm_subset = None

outdir = Path("analysis_outputs/sample_reconstructions")
outdir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Reference signal
# ------------------------------------------------------------
x_ref = get_reference_signal(all_storms, storm_num, parameter).astype(float)
t = np.arange(len(x_ref))

rng = np.random.default_rng(rep_seed)
x_deg, mask = DEGRADATION_FUNCTIONS[scenario](x_ref.copy(), severity, rng)

print("Signal length:", len(x_ref))
print("Scenario:", scenario)
print("Severity:", severity)
print("Parameter:", parameter)
print("Storm:", storm_label)
print("Number affected:", np.sum(mask) if mask is not None else "N/A")

# ------------------------------------------------------------
# Helper: extract prediction from functions that may also
# return diagnostics
# ------------------------------------------------------------
def extract_prediction(result):
    return result[0] if isinstance(result, tuple) else result

# ------------------------------------------------------------
# Get hydrologic series aligned EXACTLY with x_ref
# Only needed for missingness scenarios
# ------------------------------------------------------------
def get_aligned_hydro(all_storms, storm_num, parameter, x_ref):
    df = get_storm_dataframe(all_storms, storm_num).copy()

    pcol = find_parameter_column(df, parameter)
    if pcol is None:
        raise ValueError(f"Could not identify {parameter} column.")

    wq = pd.to_numeric(df[pcol], errors="coerce")
    valid = wq.notna()
    ref_index = df.index[valid]
    wq_check = wq.loc[valid].to_numpy(dtype=float)

    if len(wq_check) != len(x_ref):
        raise ValueError(f"WQ alignment mismatch: {len(wq_check)} vs {len(x_ref)}.")

    if not np.allclose(wq_check, x_ref, equal_nan=True):
        raise ValueError("Aligned WQ series does not match x_ref.")

    # Prefer Stage_smoothed because this matches the original
    # hydrology-informed experiment
    if "classified_storms" in globals():
        try:
            hdf = get_storm_dataframe(classified_storms, storm_num).copy()

            if "Stage_smoothed" in hdf.columns:
                stage = pd.to_numeric(hdf["Stage_smoothed"], errors="coerce")
                stage = stage.reindex(ref_index).to_numpy(dtype=float)

                if len(stage) == len(x_ref) and np.isfinite(stage).mean() > 0.95:
                    return stage, "Stage_smoothed"
        except Exception:
            pass

    # Fallback: simultaneously measured Radar from same storm
    if "Radar" not in df.columns:
        raise ValueError("Neither aligned Stage_smoothed nor Radar is available.")

    radar = pd.to_numeric(df["Radar"], errors="coerce").loc[valid].to_numpy(dtype=float)

    if len(radar) != len(x_ref):
        raise ValueError(f"Radar alignment mismatch: {len(radar)} vs {len(x_ref)}.")

    return radar, "Radar"

# ------------------------------------------------------------
# Choose algorithms
# ------------------------------------------------------------
algorithm_pool = choose_algorithm_pool(scenario).copy()

if scenario in ["random_missing", "block_missing"]:
    hydro, hydro_source = get_aligned_hydro(all_storms, storm_num, parameter, x_ref)
    print("Hydrologic auxiliary:", hydro_source)
    print("Hydrologic length:", len(hydro))
    print("Finite hydrologic values:", f"{100*np.isfinite(hydro).mean():.1f}%")

    linear_key = next((k for k in ["Linear_Interpolation", "Linear Interpolation", "Linear"] if k in algorithm_pool), None)
    if linear_key is None:
        raise KeyError(f"Linear interpolation not found. Available: {list(algorithm_pool)}")

    linear_fn = algorithm_pool[linear_key]

    algorithm_pool["Local_Moving_Mean"] = lambda x: local_moving_mean_gapfill(x=x, n_each_side=2)
    algorithm_pool["Delta_H_Weighted_Interpolation"] = lambda x: delta_h_weighted_gapfill(x=x, stage=hydro, linear_function=linear_fn)
    algorithm_pool["HPWI"] = lambda x: hpwi_gapfill(x=x, stage=hydro, linear_function=linear_fn)

if algorithm_subset is not None:
    algorithm_pool = {k: v for k, v in algorithm_pool.items() if k in algorithm_subset}

print("Algorithms to plot:", list(algorithm_pool.keys()))

# ------------------------------------------------------------
# Run reconstruction / correction
# ------------------------------------------------------------
reconstructions = {}

for alg_name, alg_func in algorithm_pool.items():
    try:
        result = alg_func(x_deg.copy())
        x_hat = np.asarray(extract_prediction(result), dtype=float)
        x_hat = x_hat[:min(len(x_ref), len(x_hat))]
        reconstructions[alg_name] = x_hat
    except Exception as e:
        print(f"{alg_name} failed: {e}")
        reconstructions[alg_name] = None

# ------------------------------------------------------------
# Plot all algorithms only
# ------------------------------------------------------------
valid_algs = [k for k, v in reconstructions.items() if v is not None]
n_algs = len(valid_algs)

ncols = 3
nrows = int(np.ceil(n_algs / ncols))

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(16, 4 * nrows),
    facecolor="white"
)

axes = np.atleast_1d(axes).flatten()

for i, alg_name in enumerate(valid_algs):
    ax = axes[i]
    x_hat = reconstructions[alg_name]
    n = min(len(t), len(x_hat))

    ax.plot(t[:n], x_ref[:n], label="Reference", linewidth=2)
    ax.plot(t[:n], x_hat[:n], label=alg_name, linewidth=1.5)

    if scenario in ["random_missing", "block_missing"]:
        missing_idx = np.where(mask[:n])[0]

        if len(missing_idx) > 0:
            ax.scatter(
                missing_idx,
                x_ref[:n][missing_idx],
                label="Missing locations",
                marker="x",
                s=20
            )

    ax.set_title(alg_name)
    ax.set_xlabel("Time index")
    ax.set_ylabel(parameter)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

# Hide unused axes if necessary
for j in range(n_algs, len(axes)):
    axes[j].axis("off")

fig.suptitle(
    f"{parameter} reconstruction | {storm_label} | {scenario} | {severity}",
    fontsize=14,
    fontweight="bold"
)

plt.tight_layout(rect=[0, 0, 1, 0.97])

save_path = outdir / f"sample_reconstruction_grid_{scenario}_{parameter}_{storm_label}_{severity}.png"

plt.savefig(
    save_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print("Saved:", save_path)


In [ ]:
# Source development-notebook cell 87
# Choose a few algorithms to overlay
best_algs = valid_algs[:4]   # or set manually, e.g. ["Wavelet", "CEEMDAN", "Median_Filter", "Savitzky_Golay"]

plt.figure(figsize=(12, 6), facecolor="white")
ax = plt.gca()
ax.set_facecolor("white")

ax.plot(t, x_ref, label="Reference", linewidth=2)
ax.plot(t, x_deg, label="Degraded", linewidth=1.2)

for alg_name in best_algs:
    x_hat = reconstructions[alg_name]
    n = min(len(t), len(x_hat))
    ax.plot(t[:n], x_hat[:n], label=alg_name, linewidth=1.5)

if scenario in ["random_missing", "block_missing"]:
    missing_idx = np.where(mask)[0]
    if len(missing_idx) > 0:
        ax.scatter(missing_idx, x_ref[missing_idx], label="Missing locations", marker="x", s=20)

ax.set_title(f"Sample correction comparison/n{parameter} | {storm_id} | {scenario} | {severity}")
ax.set_xlabel("Time index")
ax.set_ylabel(parameter)
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
save_path_overlay = outdir / f"sample_reconstruction_overlay_{scenario}_{parameter}_{storm_id}_{severity}.png"
plt.savefig(save_path_overlay, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

print("Saved:", save_path_overlay)


## Practical accuracy figures


In [ ]:
# Source development-notebook cell 104
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ----------------------------------------------------
# Load practical USGS rating summary
# ----------------------------------------------------
df = pd.read_csv("practical_usgs_rating_summary.csv")


# =========================================================
# Short display names for algorithms
# =========================================================
algo_display = {
    "Linear_Interpolation": "Linear",
    "PCHIP": "PCHIP",
    "Cubic_Spline": "Spline",
    "Kalman_Filter": "Kalman",
    "SSA_Gap_Filling": "SSA gap",

    # New gap-filling algorithms
    "Local_Moving_Mean": "Moving mean",
    "Delta_H_Weighted_Interpolation": "ΔH weighted",
    "HPWI": "HPWI",

    # Denoising algorithms
    "Median_Filter": "Median",
    "Savitzky_Golay": "Savitzky–Golay",
    "Wavelet": "Wavelet",
    "CEEMDAN": "CEEMDAN",
    "LOESS": "LOESS",
    "EWMA": "EWMA",
    "VMD": "VMD",
    "SSA": "SSA"
}

# ----------------------------------------------------
# Basic cleaning
# ----------------------------------------------------
df["scenario"] = df["scenario"].astype(str)
df["parameter"] = df["parameter"].astype(str)
df["algorithm"] = df["algorithm"].astype(str)
df["evaluation_domain"] = df["evaluation_domain"].astype(str)

# ----------------------------------------------------
# Create rating components
# ----------------------------------------------------
df["Excellent"] = df["Excellent_rate_mean"]

df["Good_only"] = (
    df["Good_or_better_rate_mean"] - df["Excellent_rate_mean"]
)

df["Poor"] = df["Poor_rate_mean"]

df["Fair_only"] = (
    100
    - df["Good_or_better_rate_mean"]
    - df["Poor_rate_mean"]
)

# Prevent tiny negative values caused by rounding
for col in ["Excellent", "Good_only", "Fair_only", "Poor"]:
    df[col] = df[col].clip(lower=0, upper=100)

rating_colors = {
    "Excellent": "#2ca25f",   # green
    "Good_only": "#3182bd",   # blue
    "Fair_only": "#f0ad4e",   # orange/yellow
    "Poor": "#d62728"         # red
}
# ----------------------------------------------------
# Choose evaluation domain by scenario
# ----------------------------------------------------
def choose_domain_for_scenario(data, scenario):
    """
    Select the most appropriate evaluation domain for each scenario.

    Missingness scenarios: use corrupted_only if available.
    Additive noise: use corrupted_only if available, otherwise full.
    Field-realistic fault: use full if available.
    """

    scenario_data = data[data["scenario"] == scenario]
    available_domains = set(scenario_data["evaluation_domain"].unique())

    if scenario in ["random_missing", "block_missing"]:
        if "corrupted_only" in available_domains:
            return "corrupted_only"
        elif "gap_only" in available_domains:
            return "gap_only"
        else:
            return sorted(available_domains)[0]

    elif scenario == "additive_noise":
        if "corrupted_only" in available_domains:
            return "corrupted_only"
        elif "full" in available_domains:
            return "full"
        else:
            return sorted(available_domains)[0]

    elif scenario == "field_realistic_fault":
        if "full" in available_domains:
            return "full"
        else:
            return sorted(available_domains)[0]

    else:
        return sorted(available_domains)[0]


plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Times New Roman", "Times", "DejaVu Serif"]
plt.rcParams["mathtext.fontset"] = "stix"
# ----------------------------------------------------
# Plot settings
# ----------------------------------------------------
scenario_order = [
    "random_missing",
    "block_missing",
    "additive_noise",
    "field_realistic_fault"
]

parameter_order = ["DO", "EC", "Temp", "Turbidity"]

rating_cols = ["Excellent", "Good_only", "Fair_only", "Poor"]
rating_labels = ["Excellent", "Good", "Fair", "Poor"]

output_dir = Path("usgs_rating_figures")
output_dir.mkdir(exist_ok=True)

top_n_algorithms = 9   # Change to 3 or 5 if needed

# ----------------------------------------------------
# Generate one stacked bar figure per scenario
# ----------------------------------------------------
for scenario in scenario_order:

    if scenario not in df["scenario"].unique():
        print(f"Skipping {scenario}: not found in data")
        continue

    domain = choose_domain_for_scenario(df, scenario)

    scenario_df = df[
        (df["scenario"] == scenario)
        & (df["evaluation_domain"] == domain)
    ].copy()

    if scenario_df.empty:
        print(f"Skipping {scenario}: no data for selected domain {domain}")
        continue

    fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharey=True)
    axes = axes.flatten()

    for ax, parameter in zip(axes, parameter_order):

        p_df = scenario_df[scenario_df["parameter"] == parameter].copy()

        if p_df.empty:
            ax.set_title(f"{parameter}: no data")
            ax.axis("off")
            continue

        # Select best algorithms by lowest mean accuracy rating
        # Lower Mean_accuracy_rating is better:
        # Excellent = 1, Good = 2, Fair = 3, Poor = 4
        p_df = (
            p_df
            .sort_values("Mean_accuracy_rating", ascending=True)
            .head(top_n_algorithms)
        )

        # Sort so best appears first
        p_df = p_df.sort_values("Mean_accuracy_rating", ascending=True)

        x = np.arange(len(p_df))
        bottom = np.zeros(len(p_df))

        for col, label in zip(rating_cols, rating_labels):
            ax.bar(
                x,
                p_df[col].values,
                bottom=bottom,
                label=label,
                color=rating_colors[col]
            )
            bottom += p_df[col].values

        ax.set_title(parameter, y=1.06)
        ax.set_xticks(x)
        ax.set_xticklabels(
            p_df["algorithm"].values,
            rotation=15,
            ha="right"
        )
        ax.set_ylim(0, 100)
        ax.set_ylabel("Percentage of corrected observations (%)")

        # Add Good-or-better text above each bar
        for i, (_, row) in enumerate(p_df.iterrows()):
            ax.text(
                i,
                101,
                f"{row['Good_or_better_rate_mean']:.1f}%",
                ha="center",
                va="bottom",
                fontsize=11
            )

    handles, labels = axes[0].get_legend_handles_labels()

    fig.legend(
        handles,
        labels,
        loc="lower center",
        ncol=4,
        frameon=False
    )

    scenario_title = scenario.replace("_", " ").title()

    #fig.suptitle(
        #f"USGS-style accuracy rating distribution under {scenario_title}/n",
        #fontsize=16,
        #y=1.0
    #)

    fig.tight_layout(rect=[0, 0, 1, 0.95])

    png_path = output_dir / f"usgs_rating_{scenario}.png"
    pdf_path = output_dir / f"usgs_rating_{scenario}.pdf"

    fig.savefig(png_path, dpi=600, bbox_inches="tight", facecolor="white")
    fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")

    plt.show()

    print(f"Saved: {png_path}")
    print(f"Saved: {pdf_path}")


In [ ]:
# Source development-notebook cell 105
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ----------------------------------------------------
# Load practical USGS rating summary
# ----------------------------------------------------
df = pd.read_csv("practical_usgs_rating_summary.csv")


# ----------------------------------------------------
# Short display names for algorithms
# ----------------------------------------------------
algo_display = {
    "Linear_Interpolation": "Linear",
    "Linear Interpolation": "Linear",

    "Cubic_Spline": "Spline",
    "Cubic Spline": "Spline",
    "Cubic_Spline_Interpolation": "Spline",
    "Cubic Spline Interpolation": "Spline",

    "PCHIP": "PCHIP",
    
    # New gap-filling algorithms
    "Local_Moving_Mean": "Moving mean",
    "Delta_H_Weighted_Interpolation": "ΔH weighted",
    "HPWI": "HPWI",

    "Kalman_Filter": "Kalman",
    "Kalman Filter": "Kalman",
    "Local_Level_Kalman_Smoother": "Kalman",
    "Local-Level Kalman Smoother": "Kalman",

    "SSA_Gap_Filling": "SSA gap",
    "SSA Gap Filling": "SSA gap",

    "Median_Filter": "Median",
    "Median Filter": "Median",

    "Savitzky_Golay": "S-G",
    "Savitzky-Golay": "S-G",
    "Savitzky–Golay": "S-G",

    "Wavelet_Filter": "Wavelet",
    "Wavelet Filter": "Wavelet",
    "Wavelet": "Wavelet",

    "CEEMDAN": "CEEMDAN",
    "LOESS": "LOESS",
    "EWMA": "EWMA",
    "VMD": "VMD",
    "SSA": "SSA"
}


# ----------------------------------------------------
# Parameter display names
# ----------------------------------------------------
parameter_display = {
    "DO": "DO",
    "EC": "EC",
    "Temp": "Temperature",
    "Turbidity": "Turbidity"
}


# ----------------------------------------------------
# Basic cleaning
# ----------------------------------------------------
df["scenario"] = df["scenario"].astype(str)
df["parameter"] = df["parameter"].astype(str)
df["algorithm"] = df["algorithm"].astype(str)
df["evaluation_domain"] = df["evaluation_domain"].astype(str)


# ----------------------------------------------------
# Create rating components
# ----------------------------------------------------
df["Excellent"] = df["Excellent_rate_mean"]

df["Good_only"] = (
    df["Good_or_better_rate_mean"]
    - df["Excellent_rate_mean"]
)

df["Poor"] = df["Poor_rate_mean"]

df["Fair_only"] = (
    100
    - df["Good_or_better_rate_mean"]
    - df["Poor_rate_mean"]
)


# Prevent tiny negative values caused by rounding
for col in ["Excellent", "Good_only", "Fair_only", "Poor"]:
    df[col] = df[col].clip(lower=0, upper=100)


# ----------------------------------------------------
# Rating colors
# ----------------------------------------------------
rating_colors = {
    "Excellent": "#2ca25f",
    "Good_only": "#3182bd",
    "Fair_only": "#f0ad4e",
    "Poor": "#d62728"
}


# ----------------------------------------------------
# Choose evaluation domain by scenario
# ----------------------------------------------------
def choose_domain_for_scenario(data, scenario):
    """
    Select the most appropriate evaluation domain
    for each degradation scenario.

    Random/block missingness:
        corrupted_only if available.

    Additive noise:
        corrupted_only if available, otherwise full.

    Field-realistic fault:
        full if available.
    """

    scenario_data = data[
        data["scenario"] == scenario
    ]

    available_domains = set(
        scenario_data["evaluation_domain"].unique()
    )

    if scenario in ["random_missing", "block_missing"]:

        if "corrupted_only" in available_domains:
            return "corrupted_only"

        elif "gap_only" in available_domains:
            return "gap_only"

        else:
            return sorted(available_domains)[0]

    elif scenario == "additive_noise":

        if "corrupted_only" in available_domains:
            return "corrupted_only"

        elif "full" in available_domains:
            return "full"

        else:
            return sorted(available_domains)[0]

    elif scenario == "field_realistic_fault":

        if "full" in available_domains:
            return "full"

        else:
            return sorted(available_domains)[0]

    else:
        return sorted(available_domains)[0]


# ----------------------------------------------------
# Font settings
# ----------------------------------------------------
plt.rcParams["font.family"] = "serif"

plt.rcParams["font.serif"] = [
    "Times New Roman",
    "Times",
    "DejaVu Serif"
]

plt.rcParams["mathtext.fontset"] = "stix"

plt.rcParams["font.size"] = 16
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 16
plt.rcParams["xtick.labelsize"] = 16
plt.rcParams["ytick.labelsize"] = 16
plt.rcParams["legend.fontsize"] = 16


# ----------------------------------------------------
# Plot settings
# ----------------------------------------------------
scenario_order = [
    "random_missing",
    "block_missing",
    "additive_noise",
    "field_realistic_fault"
]

parameter_order = [
    "DO",
    "EC",
    "Temp",
    "Turbidity"
]

rating_cols = [
    "Excellent",
    "Good_only",
    "Fair_only",
    "Poor"
]

rating_labels = [
    "Excellent",
    "Good",
    "Fair",
    "Poor"
]


# ----------------------------------------------------
# Output directory
# ----------------------------------------------------
output_dir = Path("usgs_rating_figures")

output_dir.mkdir(
    exist_ok=True
)


# Keep up to 9 algorithms
top_n_algorithms = 9


# ----------------------------------------------------
# Generate one stacked bar figure per scenario
# ----------------------------------------------------
for scenario in scenario_order:

    if scenario not in df["scenario"].unique():

        print(
            f"Skipping {scenario}: not found in data"
        )

        continue


    # ------------------------------------------------
    # Select evaluation domain
    # ------------------------------------------------
    domain = choose_domain_for_scenario(
        df,
        scenario
    )


    scenario_df = df[
        (df["scenario"] == scenario)
        &
        (df["evaluation_domain"] == domain)
    ].copy()


    if scenario_df.empty:

        print(
            f"Skipping {scenario}: "
            f"no data for selected domain {domain}"
        )

        continue


    # ------------------------------------------------
    # Create 2 x 2 figure
    # ------------------------------------------------
    fig, axes = plt.subplots(
        2,
        2,
        figsize=(16, 10),
        sharey=True
    )

    axes = axes.flatten()


    # ------------------------------------------------
    # Plot each water-quality parameter
    # ------------------------------------------------
    for ax_index, (ax, parameter) in enumerate(
        zip(axes, parameter_order)
    ):

        p_df = scenario_df[
            scenario_df["parameter"] == parameter
        ].copy()


        if p_df.empty:

            ax.set_title(
                f"{parameter_display[parameter]}: no data",
                fontsize=15
            )

            ax.axis("off")

            continue


        # --------------------------------------------
        # Select best algorithms by lowest
        # Mean_accuracy_rating
        #
        # Excellent = 1
        # Good      = 2
        # Fair      = 3
        # Poor      = 4
        # --------------------------------------------
        p_df = (
            p_df
            .sort_values(
                "Mean_accuracy_rating",
                ascending=True
            )
            .head(top_n_algorithms)
        )


        # Best algorithm appears first
        p_df = p_df.sort_values(
            "Mean_accuracy_rating",
            ascending=True
        )


        x = np.arange(
            len(p_df)
        )

        bottom = np.zeros(
            len(p_df)
        )


        # --------------------------------------------
        # Stacked vertical bars
        # --------------------------------------------
        for col, label in zip(
            rating_cols,
            rating_labels
        ):

            ax.bar(
                x,
                p_df[col].values,
                bottom=bottom,
                label=label,
                color=rating_colors[col],
                edgecolor="none",
                linewidth=0
            )

            bottom += p_df[col].values


        # --------------------------------------------
        # Parameter title
        # --------------------------------------------
        ax.set_title(
            parameter_display[parameter],
            fontsize=15,
            fontweight="bold",
            pad=8
        )


        # --------------------------------------------
        # Abbreviated algorithm names
        # --------------------------------------------
        algorithm_labels = [
            algo_display.get(
                name,
                name.replace("_", " ")
            )
            for name in p_df["algorithm"].values
        ]


        ax.set_xticks(
            x
        )


        ax.set_xticklabels(
            algorithm_labels,
            rotation=20,
            ha="right",
            fontsize=16
        )


        # --------------------------------------------
        # Y-axis settings
        # Extra space above 100 for percentage labels
        # --------------------------------------------
        ax.set_ylim(
            0,
            106
        )


        ax.set_yticks(
            np.arange(
                0,
                101,
                20
            )
        )


        ax.tick_params(
            axis="y",
            labelsize=16
        )


        # --------------------------------------------
        # Y-axis title ONLY on left-hand panels
        #
        # axes[0] = DO
        # axes[1] = EC
        # axes[2] = Temperature
        # axes[3] = Turbidity
        #
        # This prevents the y-axis title from
        # appearing in the middle of the 2 x 2 figure.
        # --------------------------------------------
        if ax_index in [0, 2]:

            ax.set_ylabel(
                "Percentage of corrected observations (%)",
                fontsize=16
            )

        else:

            ax.set_ylabel("")


        # --------------------------------------------
        # Keep Good-or-better percentage above bars
        # --------------------------------------------
        for i, (_, row) in enumerate(
            p_df.iterrows()
        ):

            ax.text(
                i,
                101.2,
                f"{row['Good_or_better_rate_mean']:.1f}%",
                ha="center",
                va="bottom",
                fontsize=16
            )


        # --------------------------------------------
        # Remove subplot boundary lines
        # --------------------------------------------
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["bottom"].set_visible(False)
        ax.spines["left"].set_visible(False)


    # ------------------------------------------------
    # Common legend
    # ------------------------------------------------
    handles, labels = (
        axes[0]
        .get_legend_handles_labels()
    )


    fig.legend(
        handles,
        labels,
        loc="lower center",
        ncol=4,
        frameon=False,
        fontsize=16
    )


    # ------------------------------------------------
    # Layout
    # No scenario title
    # ------------------------------------------------
    fig.tight_layout(
        rect=[
            0,
            0.06,
            1,
            1
        ]
    )


    # ------------------------------------------------
    # Save figures
    # ------------------------------------------------
    png_path = (
        output_dir
        / f"usgs_rating_{scenario}.png"
    )

    pdf_path = (
        output_dir
        / f"usgs_rating_{scenario}.pdf"
    )


    fig.savefig(
        png_path,
        dpi=600,
        bbox_inches="tight",
        facecolor="white"
    )


    fig.savefig(
        pdf_path,
        bbox_inches="tight",
        facecolor="white"
    )


    # ------------------------------------------------
    # Display figure
    # ------------------------------------------------
    plt.show()


    print(
        f"Saved: {png_path}"
    )

    print(
        f"Saved: {pdf_path}"
    )


In [ ]:
# Source development-notebook cell 107
import textwrap

# ----------------------------------------------------
# Cleaner labels
# ----------------------------------------------------
algorithm_label_map = {
    "Linear_Interpolation": "Linear",
    "Cubic_Spline": "Cubic spline",
    "Kalman_Filter": "Kalman",
    "SSA_Gap_Filling": "SSA",
    "Savitzky_Golay": "Savitzky-Golay",
    "Median_Filter": "Median",
    "CEEMDAN": "CEEMDAN",
    "Wavelet": "Wavelet",
    "EWMA": "EWMA",
    "LOESS": "LOESS",
    "VMD": "VMD",
    "PCHIP": "PCHIP"
}

df["algorithm_label"] = df["algorithm"].map(algorithm_label_map).fillna(df["algorithm"])

# ----------------------------------------------------
# Generate one horizontal stacked bar figure per scenario
# ----------------------------------------------------
for scenario in scenario_order:

    if scenario not in df["scenario"].unique():
        print(f"Skipping {scenario}: not found in data")
        continue

    domain = choose_domain_for_scenario(df, scenario)

    scenario_df = df[
        (df["scenario"] == scenario)
        & (df["evaluation_domain"] == domain)
    ].copy()

    if scenario_df.empty:
        print(f"Skipping {scenario}: no data for selected domain {domain}")
        continue

    fig, axes = plt.subplots(
        2, 2,
        figsize=(16, 11),
        sharex=True,
        facecolor="white"
    )

    axes = axes.flatten()

    for ax, parameter in zip(axes, parameter_order):

        p_df = scenario_df[scenario_df["parameter"] == parameter].copy()

        if p_df.empty:
            ax.set_title(f"{parameter}: no data")
            ax.axis("off")
            continue

        # Select best algorithms by lowest mean accuracy rating
        p_df = (
            p_df
            .sort_values("Mean_accuracy_rating", ascending=True)
            .head(top_n_algorithms)
            .sort_values("Mean_accuracy_rating", ascending=False)
        )

        y = np.arange(len(p_df))
        left = np.zeros(len(p_df))

        for col, label in zip(rating_cols, rating_labels):
            ax.barh(
                y,
                p_df[col].values,
                left=left,
                label=label
            )
            left += p_df[col].values

        ax.set_title(parameter, fontsize=13, fontweight="bold")
        ax.set_yticks(y)
        ax.set_yticklabels(p_df["algorithm_label"].values, fontsize=10)
        ax.set_xlim(0, 100)
        ax.set_xlabel("Percentage of corrected observations (%)")
        ax.grid(axis="x", linestyle="--", alpha=0.3)

        # Add Good-or-better labels inside/right of bars
        for i, (_, row) in enumerate(p_df.iterrows()):
            ax.text(
                101,
                i,
                f"{row['Good_or_better_rate_mean']:.1f}% Good+",
                va="center",
                ha="left",
                fontsize=9
            )

        ax.set_xlim(0, 112)

    handles, labels = axes[0].get_legend_handles_labels()

    fig.legend(
        handles,
        labels,
        loc="lower center",
        ncol=4,
        frameon=False,
        fontsize=11
    )

    scenario_title = scenario.replace("_", " ").title()

    fig.suptitle(
        f"USGS-style accuracy rating distribution under {scenario_title}/n"
        f"Evaluation domain: {domain}",
        fontsize=16,
        fontweight="bold",
        y=0.98
    )

    fig.tight_layout(rect=[0, 0, 1, 0.91])

    png_path = output_dir / f"usgs_rating_{scenario}_horizontal.png"
    pdf_path = output_dir / f"usgs_rating_{scenario}_horizontal.pdf"

    fig.savefig(png_path, dpi=300, bbox_inches="tight", facecolor="white")
    fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")

    plt.show()

    print(f"Saved: {png_path}")
    print(f"Saved: {pdf_path}")


## Temporal-resolution figures


In [ ]:
# Source development-notebook cell 127
# ----------------------------------------------------
# Prepare data for figure
# ----------------------------------------------------
plot_df = best_table.copy()

plot_df["sampling_frequency"] = pd.Categorical(
    plot_df["sampling_frequency"],
    categories=frequency_order,
    ordered=True
)

plot_df["scenario"] = pd.Categorical(
    plot_df["scenario"],
    categories=scenario_order,
    ordered=True
)

plot_df["parameter"] = pd.Categorical(
    plot_df["parameter"],
    categories=parameter_order,
    ordered=True
)

plot_df = plot_df.sort_values(
    ["scenario", "parameter", "sampling_frequency"]
)

# ----------------------------------------------------
# Figure settings
# ----------------------------------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white"
})

# ----------------------------------------------------
# Create 2 × 2 panel figure
# ----------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=True)
axes = axes.flatten()

x_labels = [frequency_labels[f] for f in frequency_order]
x_positions = np.arange(len(frequency_order))

for ax, scenario in zip(axes, scenario_order):

    sub = plot_df[plot_df["scenario"] == scenario]

    for parameter in parameter_order:

        psub = sub[sub["parameter"] == parameter].sort_values("sampling_frequency")

        y_values = []

        for freq in frequency_order:
            row = psub[psub["sampling_frequency"] == freq]

            if len(row) == 0:
                y_values.append(np.nan)
            else:
                y_values.append(row["USGS_Good_or_Better_Rate_mean"].iloc[0])

        ax.plot(
            x_positions,
            y_values,
            marker="o",
            linewidth=2,
            label=parameter
        )

    ax.set_title(scenario_labels[scenario], fontweight="bold")
    ax.set_xticks(x_positions)
    ax.set_xticklabels(x_labels)
    ax.set_ylim(0, 105)
    ax.set_ylabel("Good-or-better rate (%)")
    ax.grid(True, axis="y", alpha=0.3)

# Common legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="lower center",
    ncol=4,
    frameon=False
)

fig.suptitle(
    "Effect of sampling frequency on USGS-style Good-or-better correction rate",
    fontsize=14,
    fontweight="bold",
    y=0.98
)

fig.tight_layout(rect=[0, 0.07, 1, 0.95])

# ----------------------------------------------------
# Save figure
# ----------------------------------------------------
png_path = output_dir / "figure_frequency_sensitivity_good_or_better.png"
pdf_path = output_dir / "figure_frequency_sensitivity_good_or_better.pdf"
svg_path = output_dir / "figure_frequency_sensitivity_good_or_better.svg"

fig.savefig(png_path, dpi=600, bbox_inches="tight")
fig.savefig(pdf_path, bbox_inches="tight")
fig.savefig(svg_path, bbox_inches="tight")

plt.show()

print("Saved figure:")
print(png_path)
print(pdf_path)
print(svg_path)


In [ ]:
# Source development-notebook cell 128
# ============================================================
# OPTION 1 — RADAR / SPIDER CHART
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Radar settings
# ------------------------------------------------------------

radar_parameters = parameter_order
n_params = len(radar_parameters)

# Angles for the parameter axes
angles = np.linspace(0, 2 * np.pi, n_params, endpoint=False).tolist()
angles += angles[:1]  # close polygon

# ------------------------------------------------------------
# Create 2 × 2 polar figure
# ------------------------------------------------------------

fig, axes = plt.subplots(
    2, 2,
    figsize=(11, 9),
    subplot_kw=dict(polar=True),
    facecolor="white"
)

axes = axes.flatten()

subplot_labels = ["(a)", "(b)", "(c)", "(d)"]

for idx, (ax, scenario) in enumerate(zip(axes, scenario_order)):

    sub = plot_df[plot_df["scenario"] == scenario]

    # Plot one polygon for each temporal resolution
    for freq in frequency_order:

        values = []

        for parameter in radar_parameters:

            row = sub[
                (sub["parameter"] == parameter) &
                (sub["sampling_frequency"] == freq)
            ]

            if len(row) == 0:
                values.append(np.nan)
            else:
                values.append(
                    row["USGS_Good_or_Better_Rate_mean"].iloc[0]
                )

        # Close polygon
        values += values[:1]

        ax.plot(
            angles,
            values,
            marker="o",
            linewidth=2,
            markersize=5,
            label=frequency_labels[freq]
        )

        # Very light fill
        ax.fill(
            angles,
            values,
            alpha=0.04
        )

    # --------------------------------------------------------
    # Parameter labels
    # --------------------------------------------------------

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(
        radar_parameters,
        fontsize=10
    )

    # --------------------------------------------------------
    # Same radial scale for EVERY panel
    # --------------------------------------------------------

    ax.set_ylim(0, 100)

    ax.set_yticks([20, 40, 60, 80, 100])
    ax.set_yticklabels(
        ["20", "40", "60", "80", "100"],
        fontsize=8
    )

    ax.set_rlabel_position(22.5)

    ax.grid(
        True,
        alpha=0.35
    )

    # Panel title
    ax.set_title(
        f"{subplot_labels[idx]} {scenario_labels[scenario]}",
        fontsize=12,
        fontweight="bold",
        pad=20
    )

# ------------------------------------------------------------
# Common legend
# ------------------------------------------------------------

handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="lower center",
    ncol=len(frequency_order),
    frameon=False,
    fontsize=10,
    bbox_to_anchor=(0.5, 0.02)
)

#fig.suptitle(
    #"Sensitivity of USGS-style good-or-better correction rate "
    #"to temporal aggregation",
    #fontsize=14,
    #fontweight="bold",
    #y=0.99
#)

fig.subplots_adjust(
    top=0.90,
    bottom=0.12,
    wspace=0.30,
    hspace=0.35
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

png_path = output_dir / "figure_frequency_sensitivity_radar.png"
pdf_path = output_dir / "figure_frequency_sensitivity_radar.pdf"
svg_path = output_dir / "figure_frequency_sensitivity_radar.svg"

fig.savefig(
    png_path,
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

fig.savefig(
    pdf_path,
    bbox_inches="tight",
    facecolor="white"
)

fig.savefig(
    svg_path,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print("Radar figure saved:")
print(png_path)
print(pdf_path)
print(svg_path)


In [ ]:
# Source development-notebook cell 129
# ============================================================
# MAIN FIGURE:
# TEMPORAL-RESOLUTION EFFECT ON GAP-FILLING ALGORITHM RANK
#
# Rows:
#   Random missingness
#   Block missingness
#
# Columns:
#   Turbidity
#   EC
#   DO
#   Temperature
#
# Cells:
#   Mean composite MAE/RMSE rank
#
# Rank 1 = best
# Rank 8 = worst
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


# ============================================================
# 1. LOAD COMBINED FREQUENCY FILE
# ============================================================

file_path = (
    "results_fixed_rule_raw_frequency_expanded_sensitivity.csv"
)

df = pd.read_csv(
    file_path
)


# ============================================================
# 2. CHECK FREQUENCY LABELS
# ============================================================

print("/nSampling frequencies in file:")

print(
    df[
        "sampling_frequency"
    ]
    .value_counts(
        dropna=False
    )
)


# ============================================================
# 3. STANDARDIZE FREQUENCY LABELS
#
# Add/remove aliases here depending on what the printout shows.
# ============================================================

frequency_map = {
    "11min_original": "11–12 min",
    "30min": "30 min",
    "1h": "1 h"
}


df[
    "temporal_resolution"
] = (
    df[
        "sampling_frequency"
    ]
    .astype(str)
    .map(
        frequency_map
    )
)


# ============================================================
# 4. VERIFY ALL THREE RESOLUTIONS WERE RECOGNIZED
# ============================================================

print("/nStandardized temporal resolutions:")

print(
    df[
        "temporal_resolution"
    ]
    .value_counts(
        dropna=False
    )
)


unrecognized = (
    df.loc[
        df[
            "temporal_resolution"
        ].isna(),
        "sampling_frequency"
    ]
    .dropna()
    .unique()
)


if len(unrecognized) > 0:

    raise ValueError(
        "Some frequency labels were not recognized: "
        f"{unrecognized.tolist()}"
    )


# ============================================================
# 5. DISPLAY ORDERS
# ============================================================

resolution_order = [
    "11–12 min",
    "30 min",
    "1 h"
]


parameter_order = [
    "Turbidity",
    "EC",
    "DO",
    "Temp"
]


parameter_display = {
    "Turbidity": "Turbidity",
    "EC": "EC",
    "DO": "DO",
    "Temp": "Temperature"
}


scenario_order = [
    "random_missing",
    "block_missing"
]


scenario_display = {
    "random_missing": "Random missingness",
    "block_missing": "Block missingness"
}


# ============================================================
# 6. STANDARDIZE ALGORITHM NAMES
# ============================================================

algorithm_name_map = {

    "Linear_Interpolation": "Linear",
    "Linear Interpolation": "Linear",
    "Linear": "Linear",

    "PCHIP": "PCHIP",
    "PCHIP_Interpolation": "PCHIP",

    "Cubic_Spline": "Spline",
    "Cubic Spline": "Spline",
    "Spline": "Spline",

    "Local_Moving_Mean": "Moving mean",
    "Local Moving Mean": "Moving mean",
    "Moving Mean": "Moving mean",
    "Moving mean": "Moving mean",

    "Delta_H_Weighted_Interpolation": "ΔH weighted",
    "Delta H Weighted Interpolation": "ΔH weighted",
    "Delta_H_Weighted": "ΔH weighted",
    "ΔH weighted": "ΔH weighted",

    "HPWI": "HPWI",

    "Kalman_Filter": "Kalman",
    "Kalman Filter": "Kalman",
    "Kalman": "Kalman",

    "SSA_Gap_Filling": "SSA gap",
    "SSA Gap Filling": "SSA gap",
    "SSA_Gap": "SSA gap",
    "SSA gap": "SSA gap"
}


algorithm_order = [
    "Linear",
    "PCHIP",
    "Spline",
    "Moving mean",
    "ΔH weighted",
    "HPWI",
    "Kalman",
    "SSA gap"
]


df[
    "algorithm_display"
] = (
    df[
        "algorithm"
    ]
    .map(
        algorithm_name_map
    )
    .fillna(
        df[
            "algorithm"
        ]
    )
)


# ============================================================
# 7. FILTER TO MISSINGNESS RESULTS
#
# Only evaluate artificially corrupted/missing observations.
# ============================================================

analysis = df[
    (
        df[
            "scenario"
        ]
        .isin(
            scenario_order
        )
    )
    &
    (
        df[
            "evaluation_domain"
        ]
        ==
        "corrupted_only"
    )
    &
    (
        df[
            "status"
        ]
        ==
        "ok"
    )
    &
    (
        df[
            "algorithm_display"
        ]
        .isin(
            algorithm_order
        )
    )
].copy()


# ============================================================
# 8. VERIFY 8 ALGORITHMS AT EACH RESOLUTION
# ============================================================

algorithm_check = (
    analysis
    .groupby(
        [
            "temporal_resolution",
            "scenario"
        ]
    )[
        "algorithm_display"
    ]
    .nunique()
)


print(
    "/nNumber of algorithms by resolution/scenario:"
)

print(
    algorithm_check
)


if not (
    algorithm_check
    ==
    8
).all():

    raise ValueError(
        "At least one temporal-resolution/scenario combination "
        "does not contain all 8 gap-filling algorithms."
    )


# ============================================================
# 9. RANK WITHIN EACH MATCHED TRIAL
#
# Lower MAE/RMSE = better.
# ============================================================

rank_group = [

    "temporal_resolution",
    "parameter",
    "storm_id",
    "scenario",
    "severity",
    "repetition",
    "seed"
]


analysis[
    "MAE_rank"
] = (
    analysis
    .groupby(
        rank_group
    )[
        "MAE"
    ]
    .rank(
        method="average",
        ascending=True
    )
)


analysis[
    "RMSE_rank"
] = (
    analysis
    .groupby(
        rank_group
    )[
        "RMSE"
    ]
    .rank(
        method="average",
        ascending=True
    )
)


# ============================================================
# 10. COMPOSITE MAE/RMSE RANK
# ============================================================

analysis[
    "Composite_rank"
] = (
    analysis[
        [
            "MAE_rank",
            "RMSE_rank"
        ]
    ]
    .mean(
        axis=1
    )
)


# ============================================================
# 11. SUMMARIZE TRIAL-LEVEL RANKS
# ============================================================

rank_summary = (
    analysis
    .groupby(
        [
            "temporal_resolution",
            "scenario",
            "parameter",
            "algorithm_display"
        ],
        as_index=False
    )
    .agg(

        Mean_Rank=(
            "Composite_rank",
            "mean"
        ),

        SD_Rank=(
            "Composite_rank",
            "std"
        ),

        Median_Rank=(
            "Composite_rank",
            "median"
        ),

        N_Trials=(
            "Composite_rank",
            "size"
        )
    )
)


# ============================================================
# 12. SAVE SUMMARY DATA
# ============================================================

output_dir = Path(
    "temporal_resolution_figures"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)


rank_summary.to_csv(
    output_dir
    /
    "temporal_resolution_mean_rank_summary_missingness.csv",
    index=False
)


# ============================================================
# 13. CALCULATE CHANGE IN MEAN RANK
#
# Positive Delta Rank = worse at coarser resolution
# Negative Delta Rank = better at coarser resolution
# ============================================================

rank_change = (
    rank_summary
    .pivot_table(

        index=[
            "scenario",
            "parameter",
            "algorithm_display"
        ],

        columns=
            "temporal_resolution",

        values=
            "Mean_Rank"
    )
    .reset_index()
)


rank_change[
    "Delta_Rank_30min"
] = (
    rank_change[
        "30 min"
    ]
    -
    rank_change[
        "11–12 min"
    ]
)


rank_change[
    "Delta_Rank_1h"
] = (
    rank_change[
        "1 h"
    ]
    -
    rank_change[
        "11–12 min"
    ]
)


rank_change.to_csv(
    output_dir
    /
    "temporal_resolution_rank_change_missingness.csv",
    index=False
)


# ============================================================
# 14. CREATE 2 × 4 HEATMAP
# ============================================================

fig, axes = plt.subplots(
    nrows=2,
    ncols=4,
    figsize=(14, 9),
    constrained_layout=True
)


vmin = 1
vmax = 8

image_for_colorbar = None


for row_idx, scenario in enumerate(
    scenario_order
):

    for col_idx, parameter in enumerate(
        parameter_order
    ):

        ax = axes[
            row_idx,
            col_idx
        ]


        subset = rank_summary[
            (
                rank_summary[
                    "scenario"
                ]
                ==
                scenario
            )
            &
            (
                rank_summary[
                    "parameter"
                ]
                ==
                parameter
            )
        ].copy()


        matrix = (
            subset
            .pivot(
                index=
                    "algorithm_display",

                columns=
                    "temporal_resolution",

                values=
                    "Mean_Rank"
            )
            .reindex(
                index=
                    algorithm_order,

                columns=
                    resolution_order
            )
        )


        values = matrix.to_numpy(
            dtype=float
        )


        image = ax.imshow(
            values,
            aspect="auto",
            vmin=vmin,
            vmax=vmax
        )


        image_for_colorbar = image


        # ----------------------------------------------------
        # X axis
        # ----------------------------------------------------

        ax.set_xticks(
            np.arange(
                len(
                    resolution_order
                )
            )
        )

        ax.set_xticklabels(
            resolution_order,
            rotation=35,
            ha="right"
        )


        # ----------------------------------------------------
        # Y axis
        # ----------------------------------------------------

        ax.set_yticks(
            np.arange(
                len(
                    algorithm_order
                )
            )
        )


        if col_idx == 0:

            ax.set_yticklabels(
                algorithm_order
            )

        else:

            ax.set_yticklabels(
                []
            )


        # ----------------------------------------------------
        # PARAMETER TITLES
        # ----------------------------------------------------

        if row_idx == 0:

            ax.set_title(
                parameter_display[
                    parameter
                ],
                fontweight="bold"
            )


        # ----------------------------------------------------
        # SCENARIO LABELS
        # ----------------------------------------------------

        if col_idx == 0:

            ax.set_ylabel(
                scenario_display[
                    scenario
                ],
                fontweight="bold"
            )


        # ----------------------------------------------------
        # GRID
        # ----------------------------------------------------

        ax.set_xticks(
            np.arange(
                -0.5,
                len(
                    resolution_order
                ),
                1
            ),
            minor=True
        )

        ax.set_yticks(
            np.arange(
                -0.5,
                len(
                    algorithm_order
                ),
                1
            ),
            minor=True
        )

        ax.grid(
            which="minor",
            linewidth=0.5
        )

        ax.tick_params(
            which="minor",
            bottom=False,
            left=False
        )


# ============================================================
# 15. COLORBAR
# ============================================================

cbar = fig.colorbar(
    image_for_colorbar,
    ax=axes,
    fraction=0.025,
    pad=0.02
)


cbar.set_label(
    "Mean composite rank (1 = best, 8 = worst)"
)


cbar.set_ticks(
    np.arange(
        1,
        9
    )
)


# ============================================================
# 16. MAIN TITLE
# ============================================================

fig.suptitle(
    "Effect of temporal aggregation on gap-filling "
    "algorithm performance",
    fontsize=14,
    fontweight="bold"
)


# ============================================================
# 17. SAVE
# ============================================================

png_path = (
    output_dir
    /
    "temporal_resolution_mean_rank_missingness.png"
)

pdf_path = (
    output_dir
    /
    "temporal_resolution_mean_rank_missingness.pdf"
)


fig.savefig(
    png_path,
    dpi=600,
    bbox_inches="tight"
)


fig.savefig(
    pdf_path,
    bbox_inches="tight"
)


plt.show()


print(
    "/nSaved:"
)

print(
    png_path
)

print(
    pdf_path
)


In [ ]:
# Source development-notebook cell 130
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Load
df = pd.read_csv("results_fixed_rule_raw_frequency_expanded_sensitivity.csv")

# Frequency labels
freq_map = {"11min_original":"11–12 min", "30min":"30 min", "1h":"1 h"}
df["resolution"] = df["sampling_frequency"].map(freq_map)

# Scenario/parameter order
scenario_order = ["random_missing", "block_missing", "additive_noise", "field_realistic_fault"]
scenario_names = {"random_missing":"Random missingness", "block_missing":"Block missingness",
                  "additive_noise":"Additive noise", "field_realistic_fault":"Field-realistic fault"}
parameter_order = ["Turbidity", "EC", "DO", "Temp"]
parameter_names = {"Turbidity":"Turbidity", "EC":"EC", "DO":"DO", "Temp":"Temperature"}
resolution_order = ["11–12 min", "30 min", "1 h"]

# Display names for gap-filling algorithms; all others are cleaned automatically
name_map = {
    "Linear_Interpolation":"Linear", "PCHIP":"PCHIP", "Cubic_Spline":"Spline",
    "Local_Moving_Mean":"Moving mean", "Delta_H_Weighted_Interpolation":"ΔH weighted",
    "HPWI":"HPWI", "Kalman_Filter":"Kalman", "SSA_Gap_Filling":"SSA gap"
}
df["algorithm_display"] = df["algorithm"].map(name_map).fillna(df["algorithm"].str.replace("_", " ", regex=False))

# Check evaluation domains before filtering
print(df.groupby("scenario")["evaluation_domain"].value_counts())

# Missingness = corrupted points only; noise/fault = full signal
#mask_missing = df["scenario"].isin(["random_missing","block_missing"]) & df["evaluation_domain"].eq("corrupted_only")
#mask_other = df["scenario"].isin(["additive_noise","field_realistic"]) & df["evaluation_domain"].eq("full")
#a = df[df["status"].eq("ok") & (mask_missing | mask_other)].copy()
missing = ["random_missing", "block_missing"]
other = ["additive_noise", "field_realistic"]

mask_missing = df["scenario"].isin(missing) & df["evaluation_domain"].eq("corrupted_only")
mask_other = df["scenario"].isin(other) & ~df["evaluation_domain"].isin(["corrupted_only", "error"])
a = df[df["status"].eq("ok") & (mask_missing | mask_other)].copy()

# Check algorithm counts
print("/nAlgorithms per scenario and resolution:")
print(a.groupby(["scenario","resolution"])["algorithm_display"].nunique())

# Rank MAE and RMSE within each matched trial
rank_cols = ["resolution","parameter","storm_id","scenario","severity","repetition","seed"]
a["MAE_rank"] = a.groupby(rank_cols)["MAE"].rank(method="average", ascending=True)
a["RMSE_rank"] = a.groupby(rank_cols)["RMSE"].rank(method="average", ascending=True)
a["Composite_rank"] = a[["MAE_rank","RMSE_rank"]].mean(axis=1)

# Mean trial-level rank
summary = a.groupby(["resolution","scenario","parameter","algorithm_display"], as_index=False).agg(
    Mean_Rank=("Composite_rank","mean"),
    SD_Rank=("Composite_rank","std"),
    N=("Composite_rank","size")
)

# Algorithm order within each scenario: best overall mean rank first
algorithm_orders = {}
for scenario in scenario_order:
    s = summary[summary["scenario"].eq(scenario)]
    algorithm_orders[scenario] = s.groupby("algorithm_display")["Mean_Rank"].mean().sort_values().index.tolist()

print("/nAlgorithm order by scenario:")
for scenario, algorithms in algorithm_orders.items():
    print(scenario, algorithms)

# Save summary
out = Path("temporal_resolution_figures")
out.mkdir(exist_ok=True)
summary.to_csv(out/"temporal_resolution_rank_summary_all_scenarios.csv", index=False)

# Plot
fig, axes = plt.subplots(4, 4, figsize=(15, 15), constrained_layout=True)
max_rank = max(len(v) for v in algorithm_orders.values())

for r, scenario in enumerate(scenario_order):
    algorithms = algorithm_orders[scenario]

    for c, parameter in enumerate(parameter_order):
        ax = axes[r, c]
        s = summary[(summary["scenario"] == scenario) & (summary["parameter"] == parameter)]
        matrix = s.pivot(index="algorithm_display", columns="resolution", values="Mean_Rank")
        matrix = matrix.reindex(index=algorithms, columns=resolution_order)

        im = ax.imshow(matrix.values, aspect="auto", vmin=1, vmax=max_rank)

        ax.set_xticks(range(3))
        ax.set_xticklabels(resolution_order, rotation=35, ha="right")
        ax.set_yticks(range(len(algorithms)))

        if c == 0:
            ax.set_yticklabels(algorithms)
            ax.set_ylabel(scenario_names[scenario], fontweight="bold")
        else:
            ax.set_yticklabels([])

        if r == 0:
            ax.set_title(parameter_names[parameter], fontweight="bold")

        ax.set_xticks(np.arange(-0.5, 3, 1), minor=True)
        ax.set_yticks(np.arange(-0.5, len(algorithms), 1), minor=True)
        ax.grid(which="minor", linewidth=0.5)
        ax.tick_params(which="minor", bottom=False, left=False)

cbar = fig.colorbar(im, ax=axes, fraction=0.02, pad=0.02)
cbar.set_label(f"Mean composite rank (1 = best, {max_rank} = worst)")
cbar.set_ticks(range(1, max_rank + 1))

fig.suptitle("Effect of temporal aggregation on algorithm performance", fontsize=15, fontweight="bold")
fig.savefig(out/"temporal_resolution_rank_all_scenarios.png", dpi=600, bbox_inches="tight")
fig.savefig(out/"temporal_resolution_rank_all_scenarios.pdf", bbox_inches="tight")
plt.show()


In [ ]:
# Source development-notebook cell 131
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Load data
df = pd.read_csv("results_fixed_rule_raw_frequency_expanded_sensitivity.csv")
df["resolution"] = df["sampling_frequency"].map({"11min_original":"11–12 min", "30min":"30 min", "1h":"1 h"})

# Display names
name_map = {
    "Linear_Interpolation":"Linear", "Cubic_Spline":"Spline", "Kalman_Filter":"Kalman",
    "SSA_Gap_Filling":"SSA gap", "Local_Moving_Mean":"Moving mean",
    "Delta_H_Weighted_Interpolation":"ΔH weighted", "HPWI":"HPWI",
    "Savitzky_Golay":"Savitzky–Golay", "Median_Filter":"Median filter"
}
df["algorithm_display"] = df["algorithm"].map(name_map).fillna(df["algorithm"].str.replace("_", " ", regex=False))

scenario_order = ["random_missing", "block_missing", "additive_noise", "field_realistic_fault"]

scenario_names = {
    "random_missing": "Random missingness",
    "block_missing": "Block missingness",
    "additive_noise": "Additive noise",
    "field_realistic_fault": "Field-realistic fault"
}
parameter_order = ["Turbidity", "EC", "DO", "Temp"]
parameter_names = {"Turbidity":"Turbidity", "EC":"EC", "DO":"DO", "Temp":"Temperature"}
resolution_order = ["11–12 min", "30 min", "1 h"]

# Check what is actually in the file
print("Scenarios:", df["scenario"].dropna().unique())
print("/nEvaluation domains:")
print(df.groupby("scenario")["evaluation_domain"].value_counts())

# Automatically identify evaluation domain for each scenario
domain_map = {
    "random_missing": "corrupted_only",
    "block_missing": "corrupted_only",
    "additive_noise": "full",
    "field_realistic_fault": "full"
}

parts = []
for scenario in scenario_order:
    s = df[(df["scenario"] == scenario) &
           (df["status"] == "ok") &
           (df["evaluation_domain"] == domain_map[scenario])].copy()
    parts.append(s)

a = pd.concat(parts, ignore_index=True)

print("/nAlgorithm counts:")
print(a.groupby(["scenario", "resolution"])["algorithm_display"].nunique())

# Rank MAE and RMSE within each trial
rank_cols = ["resolution", "parameter", "storm_id", "scenario", "severity", "repetition", "seed"]
a["MAE_rank"] = a.groupby(rank_cols)["MAE"].rank(method="average", ascending=True)
a["RMSE_rank"] = a.groupby(rank_cols)["RMSE"].rank(method="average", ascending=True)
a["Composite_rank"] = a[["MAE_rank", "RMSE_rank"]].mean(axis=1)

# Mean rank across trials
summary = a.groupby(["scenario", "parameter", "resolution", "algorithm_display"], as_index=False).agg(
    Mean_Rank=("Composite_rank", "mean"),
    SD_Rank=("Composite_rank", "std"),
    N=("Composite_rank", "size")
)

out = Path("temporal_resolution_trajectory")
out.mkdir(exist_ok=True)
summary.to_csv(out/"temporal_resolution_rank_trajectory_summary.csv", index=False)

# Plot one figure per scenario
x = np.arange(len(resolution_order))

for scenario in scenario_order:
    ssc = summary[summary["scenario"] == scenario].copy()
    algorithms = ssc.groupby("algorithm_display")["Mean_Rank"].mean().sort_values().index.tolist()
    max_rank = len(algorithms)

    fig, axes = plt.subplots(1, 4, figsize=(15, 5), sharey=True)

    for ax, parameter in zip(axes, parameter_order):
        sp = ssc[ssc["parameter"] == parameter]

        for alg in algorithms:
            line = sp[sp["algorithm_display"] == alg].set_index("resolution").reindex(resolution_order)
            y = line["Mean_Rank"].to_numpy(dtype=float)
            ax.plot(x, y, marker="o", linewidth=1.8, label=alg)

        ax.set_title(parameter_names[parameter], fontweight="bold")
        ax.set_xticks(x)
        ax.set_xticklabels(resolution_order)
        ax.set_xlabel("Temporal resolution")
        ax.set_ylim(max_rank + 0.5, 0.5)
        ax.set_yticks(range(1, max_rank + 1))
        ax.grid(alpha=0.3)

    axes[0].set_ylabel("Mean composite rank (1 = best)")

    handles, labels = axes[-1].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=min(5, len(algorithms)), bbox_to_anchor=(0.5, -0.08))

    fig.suptitle(f"{scenario_names[scenario]}: algorithm response to temporal aggregation", fontweight="bold")
    fig.tight_layout(rect=[0, 0.10, 1, 0.93])

    filename = scenario.replace("_", "-")
    fig.savefig(out/f"rank_trajectory_{filename}.png", dpi=600, bbox_inches="tight")
    fig.savefig(out/f"rank_trajectory_{filename}.pdf", bbox_inches="tight")
    plt.show()


In [ ]:
# Source development-notebook cell 132
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

df = pd.read_csv("results_fixed_rule_raw_frequency_expanded_sensitivity.csv")
df["resolution"] = df["sampling_frequency"].map({"11min_original":"11–12 min","30min":"30 min","1h":"1 h"})

name_map = {
    "Linear_Interpolation":"Linear","Cubic_Spline":"Spline","Kalman_Filter":"Kalman",
    "SSA_Gap_Filling":"SSA gap","Local_Moving_Mean":"Moving mean",
    "Delta_H_Weighted_Interpolation":"ΔH weighted","HPWI":"HPWI",
    "Savitzky_Golay":"Savitzky–Golay","Median_Filter":"Median filter"
}
df["algorithm_display"] = df["algorithm"].map(name_map).fillna(df["algorithm"].str.replace("_"," ",regex=False))

scenarios = ["random_missing","block_missing","additive_noise","field_realistic_fault"]
scenario_names = {
    "random_missing":"Random missingness","block_missing":"Block missingness",
    "additive_noise":"Additive noise","field_realistic_fault":"Field-realistic fault"
}
parameters = ["Turbidity","EC","DO","Temp"]
parameter_names = {"Turbidity":"Turbidity","EC":"EC","DO":"DO","Temp":"Temperature"}
resolutions = ["11–12 min","30 min","1 h"]
domain_map = {"random_missing":"corrupted_only","block_missing":"corrupted_only",
              "additive_noise":"full","field_realistic_fault":"full"}

# Select correct evaluation domain
parts = []
for scenario in scenarios:
    parts.append(df[(df["scenario"]==scenario) & (df["status"]=="ok") &
                    (df["evaluation_domain"]==domain_map[scenario])].copy())
a = pd.concat(parts, ignore_index=True)

# Mean MAE/RMSE across storms, severity levels and repetitions
summary_error = a.groupby(
    ["scenario","parameter","resolution","algorithm_display"], as_index=False
).agg(
    MAE=("MAE","mean"), RMSE=("RMSE","mean"),
    MAE_SD=("MAE","std"), RMSE_SD=("RMSE","std"), N=("MAE","size")
)

summary_error.to_csv("temporal_resolution_absolute_error_summary.csv", index=False)

# Plot
out = Path("temporal_resolution_absolute_error")
out.mkdir(exist_ok=True)
x = np.arange(3)

for scenario in scenarios:
    ssc = summary_error[summary_error["scenario"]==scenario]
    algorithms = sorted(ssc["algorithm_display"].unique())

    fig, axes = plt.subplots(2,4,figsize=(15,8))

    for c, parameter in enumerate(parameters):
        sp = ssc[ssc["parameter"]==parameter]

        for alg in algorithms:
            line = sp[sp["algorithm_display"]==alg].set_index("resolution").reindex(resolutions)
            axes[0,c].plot(x,line["MAE"].to_numpy(float),marker="o",linewidth=1.6,label=alg)
            axes[1,c].plot(x,line["RMSE"].to_numpy(float),marker="o",linewidth=1.6,label=alg)

        axes[0,c].set_title(parameter_names[parameter],fontweight="bold")

        for r in range(2):
            axes[r,c].set_xticks(x)
            axes[r,c].set_xticklabels(resolutions,rotation=25)
            axes[r,c].grid(alpha=.25)

    axes[0,0].set_ylabel("Mean MAE")
    axes[1,0].set_ylabel("Mean RMSE")

    handles, labels = axes[0,-1].get_legend_handles_labels()
    fig.legend(handles,labels,loc="lower center",ncol=min(5,len(algorithms)),bbox_to_anchor=(.5,-.01))
    fig.suptitle(f"{scenario_names[scenario]}: absolute error response to temporal aggregation",fontweight="bold")
    fig.tight_layout(rect=[0,.08,1,.94])

    name = scenario.replace("_","-")
    fig.savefig(out/f"MAE_RMSE_{name}.png",dpi=600,bbox_inches="tight")
    fig.savefig(out/f"MAE_RMSE_{name}.pdf",bbox_inches="tight")
    plt.show()


In [ ]:
# Source development-notebook cell 133
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("results_fixed_rule_raw_frequency_expanded_sensitivity.csv")

df["resolution"] = df["sampling_frequency"].map({
    "11min_original": "11–12 min",
    "30min": "30 min",
    "1h": "1 h"
})

algo_map = {
    "Linear_Interpolation": "Linear",
    "PCHIP": "PCHIP",
    "Cubic_Spline": "Spline",
    "Kalman_Filter": "Kalman",
    "SSA_Gap_Filling": "SSA gap",
    "Local_Moving_Mean": "Moving mean",
    "Delta_H_Weighted_Interpolation": "ΔH weighted",
    "HPWI": "HPWI",
    "Median_Filter": "Median filter",
    "Savitzky_Golay": "Savitzky–Golay",
    "LOESS": "LOESS",
    "SSA": "SSA",
    "Wavelet": "Wavelet",
    "VMD": "VMD",
    "EWMA": "EWMA",
    "CEEMDAN": "CEEMDAN"
}

scenario_order = ["random_missing", "block_missing", "additive_noise", "field_realistic_fault"]

scenario_labels = {
    "random_missing": "Random missingness",
    "block_missing": "Block missingness",
    "additive_noise": "Additive noise",
    "field_realistic_fault": "Field-realistic fault"
}

parameter_order = ["Turbidity", "EC", "DO", "Temp"]

parameter_labels = {
    "Turbidity": "Turbidity",
    "EC": "EC",
    "DO": "DO",
    "Temp": "Temperature"
}

resolution_order = ["11–12 min", "30 min", "1 h"]

domain_map = {
    "random_missing": "corrupted_only",
    "block_missing": "corrupted_only",
    "additive_noise": "full",
    "field_realistic_fault": "full"
}

df["algorithm_display"] = df["algorithm"].map(algo_map).fillna(df["algorithm"])

plot_df = df[df["status"] == "ok"].copy()
plot_df = plot_df[
    plot_df.apply(lambda r: r["evaluation_domain"] == domain_map.get(r["scenario"]), axis=1)
].copy()

metric = "RMSE"

summary = plot_df.groupby(
    ["scenario", "parameter", "resolution", "algorithm_display"], as_index=False
)[metric].mean()

all_algorithms = list(summary["algorithm_display"].dropna().unique())
x = np.arange(len(resolution_order))

# ============================================================
# FIGURE LAYOUT
#
# Grid rows:
# 0 = Random missingness
# 1 = Block missingness
# 2 = Gap-filling legend
# 3 = Additive noise
# 4 = Field-realistic fault
# 5 = Noise/fault legend
# ============================================================

fig = plt.figure(figsize=(16, 15))

gs = fig.add_gridspec(
    6, 4,
    height_ratios=[1, 1, 0.16, 1, 1, 0.16],
    hspace=0.38,
    wspace=0.25
)

axes = np.empty((4, 4), dtype=object)
grid_rows = [0, 1, 3, 4]

for i in range(4):
    for j in range(4):
        axes[i, j] = fig.add_subplot(gs[grid_rows[i], j])

gap_legend_ax = fig.add_subplot(gs[2, :])
gap_legend_ax.axis("off")

correction_legend_ax = fig.add_subplot(gs[5, :])
correction_legend_ax.axis("off")

# ============================================================
# PLOT
# ============================================================

for i, scenario in enumerate(scenario_order):
    for j, parameter in enumerate(parameter_order):
        ax = axes[i, j]
        sub = summary[(summary["scenario"] == scenario) & (summary["parameter"] == parameter)]

        for alg in all_algorithms:
            s = sub[sub["algorithm_display"] == alg].set_index("resolution").reindex(resolution_order)
            y = s[metric].to_numpy(dtype=float)

            if np.isfinite(y).any():
                ax.plot(x, y, marker="o", linewidth=1.6, label=alg)

        # Parameter titles at beginning of each algorithm class
        if i in [0, 2]:
            ax.set_title(parameter_labels[parameter], fontweight="bold")

        if j == 0:
            ax.set_ylabel(f"{scenario_labels[scenario]}/nMean {metric}")

        # Show temporal-resolution labels only after each class
        ax.set_xticks(x)

        if i in [1, 3]:
            ax.set_xticklabels(resolution_order, rotation=0)
            ax.set_xlabel("Temporal resolution")
        else:
            ax.set_xticklabels([])

        ax.grid(alpha=0.25)

# ============================================================
# LEGEND HELPER
# ============================================================

def collect_unique_handles(ax_group):
    out = {}

    for ax in ax_group:
        handles, labels = ax.get_legend_handles_labels()

        for handle, label in zip(handles, labels):
            if label and label not in out:
                out[label] = handle

    return out

# ============================================================
# GAP-FILLING LEGEND
# ============================================================

gap_handles = collect_unique_handles(axes[:2, :].ravel())

print("Gap algorithms:", list(gap_handles.keys()))

gap_legend_ax.legend(
    gap_handles.values(),
    gap_handles.keys(),
    title="Gap-filling algorithms",
    loc="center",
    ncol=4,
    frameon=False
)

# ============================================================
# NOISE / FAULT LEGEND
# ============================================================

correction_handles = collect_unique_handles(axes[2:, :].ravel())

print("Noise/fault algorithms:", list(correction_handles.keys()))

correction_legend_ax.legend(
    correction_handles.values(),
    correction_handles.keys(),
    title="Noise/fault-correction algorithms",
    loc="center",
    ncol=5,
    frameon=False
)

# ============================================================
# FINISH
# ============================================================

#fig.suptitle(
   # f"Effect of temporal aggregation on algorithm performance ({metric})",
   # fontweight="bold",
   # fontsize=14,
   # y=0.995
#)

fig.savefig(
    f"figure_temporal_aggregation_{metric}.png",
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    f"figure_temporal_aggregation_{metric}.pdf",
    bbox_inches="tight"
)

plt.show()


In [ ]:
# Source development-notebook cell 134
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("results_fixed_rule_raw_frequency_expanded_sensitivity.csv")

df["resolution"] = df["sampling_frequency"].map({
    "11min_original":"11–12 min", "30min":"30 min", "1h":"1 h"
})

algo_map = {
    "Linear_Interpolation":"Linear", "PCHIP":"PCHIP", "Cubic_Spline":"Spline",
    "Kalman_Filter":"Kalman", "SSA_Gap_Filling":"SSA gap",
    "Local_Moving_Mean":"Moving mean", "Delta_H_Weighted_Interpolation":"ΔH weighted",
    "HPWI":"HPWI", "Median_Filter":"Median filter",
    "Savitzky_Golay":"Savitzky–Golay", "LOESS":"LOESS", "SSA":"SSA",
    "Wavelet":"Wavelet", "VMD":"VMD", "EWMA":"EWMA", "CEEMDAN":"CEEMDAN"
}

df["algorithm_display"] = df["algorithm"].map(algo_map).fillna(df["algorithm"])

scenarios = ["random_missing","block_missing","additive_noise","field_realistic_fault"]
scenario_labels = {
    "random_missing":"Random missingness", "block_missing":"Block missingness",
    "additive_noise":"Additive noise", "field_realistic_fault":"Field-realistic fault"
}

parameters = ["Turbidity","EC","DO","Temp"]
parameter_labels = {"Turbidity":"Turbidity","EC":"Electrical conductivity","DO":"Dissolved oxygen","Temp":"Temperature"}

domain_map = {
    "random_missing":"corrupted_only", "block_missing":"corrupted_only",
    "additive_noise":"full", "field_realistic_fault":"full"
}

# Correct evaluation domain
parts = []
for scenario in scenarios:
    parts.append(df[(df["scenario"]==scenario) & (df["status"]=="ok") &
                    (df["evaluation_domain"]==domain_map[scenario])].copy())

a = pd.concat(parts, ignore_index=True)

# Mean RMSE at each temporal resolution
mean_rmse = a.groupby(
    ["scenario","parameter","algorithm_display","resolution"], as_index=False
)["RMSE"].mean()

# Put resolutions side-by-side
wide = mean_rmse.pivot(
    index=["scenario","parameter","algorithm_display"],
    columns="resolution", values="RMSE"
).reset_index()

wide = wide.dropna(subset=["11–12 min","30 min","1 h"])
wide = wide[wide["11–12 min"] > 0].copy()

# % change based on GROUP MEAN RMSE, not mean trial-level percentages
wide["30 min"] = 100 * (wide["30 min"] - wide["11–12 min"]) / wide["11–12 min"]
wide["1 h"] = 100 * (wide["1 h"] - wide["11–12 min"]) / wide["11–12 min"]

# Preferred algorithm order
gap_order = ["Linear","PCHIP","Spline","Kalman","SSA gap","Moving mean","ΔH weighted","HPWI"]
correction_order = ["Median filter","LOESS","SSA","Savitzky–Golay","Kalman",
                    "Wavelet","VMD","EWMA","CEEMDAN"]

fig, axes = plt.subplots(4,4,figsize=(14,14))
bar_h = 0.34

for i, scenario in enumerate(scenarios):
    preferred = gap_order if scenario in ["random_missing","block_missing"] else correction_order

    for j, parameter in enumerate(parameters):
        ax = axes[i,j]
        sub = wide[(wide["scenario"]==scenario) & (wide["parameter"]==parameter)].copy()

        # Keep every algorithm actually present
        present = sub["algorithm_display"].unique()
        algorithms = [alg for alg in preferred if alg in present]
        algorithms += [alg for alg in present if alg not in algorithms]

        sub = sub.set_index("algorithm_display").reindex(algorithms)
        y = np.arange(len(algorithms))

        ax.barh(y-bar_h/2, sub["30 min"], height=bar_h, label="30 min")
        ax.barh(y+bar_h/2, sub["1 h"], height=bar_h, label="1 h")
        ax.axvline(0, color="black", linewidth=0.8)

        # PANEL-SPECIFIC symmetric scale
        vals = np.abs(sub[["30 min","1 h"]].to_numpy(dtype=float))
        limit = np.nanmax(vals) if np.isfinite(vals).any() else 1
        limit = max(5, np.ceil(limit/5)*5)
        ax.set_xlim(-limit, limit)

        ax.set_yticks(y)
        if j == 0:
            ax.set_yticklabels(algorithms)
            ax.set_ylabel(scenario_labels[scenario], fontweight="bold", fontsize=12)
        else:
            ax.set_yticklabels([])

        ax.invert_yaxis()
        ax.grid(axis="x", alpha=0.25)
        # Remove subplot borders
        #for spine in ax.spines.values():
            #spine.set_visible(False)

        if i == 0:
            ax.set_title(parameter_labels[parameter], fontweight="bold")

        if i == 3:
            ax.set_xlabel("Change in RMSE (%)")

handles, labels = axes[0,0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=2, frameon=False)

#fig.suptitle(
   # "Effect of temporal aggregation on RMSE relative to the original 11–12 min resolution",
   # fontsize=13, fontweight="bold"
#)

fig.text(
    0.5, 0.025,
    "Negative values indicate lower RMSE; positive values indicate higher RMSE after temporal aggregation.",
    ha="center"
)

fig.tight_layout(rect=[0,0.05,1,0.96])

fig.savefig("figure_temporal_aggregation_RMSE_change.png", dpi=600, bbox_inches="tight")
fig.savefig("figure_temporal_aggregation_RMSE_change.pdf", bbox_inches="tight")

plt.show()


In [ ]:
# Source development-notebook cell 136
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import matplotlib.dates as mdates

# ----------------------------------------------------
# Settings
# ----------------------------------------------------
parameters = ["Turbidity", "EC", "DO", "Temp"]

parameter_titles = {
    "Turbidity": "Turbidity (NTU)",
    "EC": "Specific Conductivity",
    "DO": "Dissolved Oxygen",
    "Temp": "Water Temperature (°C)"
}

storm_ids = sorted(all_storms.keys())

output_dir = Path("supplementary_resampled_signal_plots")
output_dir.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------
# Plot formatting
# ----------------------------------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white"
})

# ----------------------------------------------------
# Helper function for safe plotting
# ----------------------------------------------------
def safe_plot(ax, df, param, label, linewidth=1.4, marker=None, markersize=3, alpha=0.9):
    """
    Plot pandas datetime index and parameter values safely by converting to numpy arrays.
    """

    if param not in df.columns:
        return

    temp = df[[param]].copy()
    temp = temp.dropna()

    if temp.empty:
        return

    x = pd.to_datetime(temp.index).to_numpy()
    y = temp[param].astype(float).to_numpy()

    ax.plot(
        x,
        y,
        label=label,
        linewidth=linewidth,
        marker=marker,
        markersize=markersize,
        alpha=alpha
    )

# ----------------------------------------------------
# Create one figure per storm
# ----------------------------------------------------
for storm_id in storm_ids:

    df_orig = all_storms[storm_id]["data"].copy()
    df_30 = all_storms_30min[storm_id]["data"].copy()
    df_1h = all_storms_1h[storm_id]["data"].copy()

    # Ensure datetime index
    df_orig.index = pd.to_datetime(df_orig.index)
    df_30.index = pd.to_datetime(df_30.index)
    df_1h.index = pd.to_datetime(df_1h.index)

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    axes = axes.flatten()

    for ax, param in zip(axes, parameters):

        safe_plot(
            ax,
            df_orig,
            param,
            label="Original (~11–12 min)",
            linewidth=1.4,
            marker=None,
            alpha=0.9
        )

        safe_plot(
            ax,
            df_30,
            param,
            label="30 min",
            linewidth=1.8,
            marker="o",
            markersize=3,
            alpha=0.9
        )

        safe_plot(
            ax,
            df_1h,
            param,
            label="1 h",
            linewidth=2.0,
            marker="s",
            markersize=4,
            alpha=0.9
        )

        ax.set_title(parameter_titles.get(param, param), fontweight="bold")
        ax.set_xlabel("Time")
        ax.set_ylabel(parameter_titles.get(param, param))
        ax.grid(True, alpha=0.3)

        # Better date formatting
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d/n%H:%M"))
        ax.tick_params(axis="x", rotation=0)

    handles, labels = axes[0].get_legend_handles_labels()

    fig.legend(
        handles,
        labels,
        loc="lower center",
        ncol=3,
        frameon=False
    )

    fig.suptitle(
        f"Storm {storm_id}: Original and Resampled Water-Quality Signals",
        fontsize=14,
        fontweight="bold",
        y=0.98
    )

    fig.tight_layout(rect=[0, 0.07, 1, 0.95])

    png_file = output_dir / f"storm_{storm_id}_resampled_signals.png"
    pdf_file = output_dir / f"storm_{storm_id}_resampled_signals.pdf"

    fig.savefig(png_file, dpi=600, bbox_inches="tight")
    fig.savefig(pdf_file, bbox_inches="tight")

    plt.show()

    print(f"Saved: {png_file}")
    print(f"Saved: {pdf_file}")


In [ ]:
# Source development-notebook cell 137
# ----------------------------------------------------
# Combined all-storm supplementary figure
# ----------------------------------------------------
n_rows = len(storm_ids)
n_cols = len(parameters)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(18, 3.8 * n_rows),
    squeeze=False
)

for i, storm_id in enumerate(storm_ids):

    df_orig = all_storms[storm_id]["data"].copy()
    df_30 = all_storms_30min[storm_id]["data"].copy()
    df_1h = all_storms_1h[storm_id]["data"].copy()

    df_orig.index = pd.to_datetime(df_orig.index)
    df_30.index = pd.to_datetime(df_30.index)
    df_1h.index = pd.to_datetime(df_1h.index)

    for j, param in enumerate(parameters):

        ax = axes[i, j]

        safe_plot(
            ax,
            df_orig,
            param,
            label="Original (~11–12 min)",
            linewidth=1.2,
            marker=None,
            alpha=0.9
        )

        safe_plot(
            ax,
            df_30,
            param,
            label="30 min",
            linewidth=1.6,
            marker="o",
            markersize=2.5,
            alpha=0.9
        )

        safe_plot(
            ax,
            df_1h,
            param,
            label="1 h",
            linewidth=1.8,
            marker="s",
            markersize=3,
            alpha=0.9
        )

        if i == 0:
            ax.set_title(parameter_titles.get(param, param), fontweight="bold")

        if j == 0:
            ax.set_ylabel(f"Storm {storm_id}")

        ax.set_xlabel("Time")
        ax.grid(True, alpha=0.25)

        ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d/n%H:%M"))

handles, labels = axes[0, 0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="lower center",
    ncol=3,
    frameon=False
)

fig.suptitle(
    "Original and Resampled Water-Quality Storm Signals Across All Storms",
    fontsize=15,
    fontweight="bold",
    y=0.995
)

fig.tight_layout(rect=[0, 0.05, 1, 0.97])

png_file = output_dir / "all_storms_resampled_signals.png"
pdf_file = output_dir / "all_storms_resampled_signals.pdf"

fig.savefig(png_file, dpi=600, bbox_inches="tight")
fig.savefig(pdf_file, bbox_inches="tight")

plt.show()

print(f"Saved: {png_file}")
print(f"Saved: {pdf_file}")


## Repetition/convergence figures


In [ ]:
# Source development-notebook cell 143
# ----------------------------------------------------
# Rank algorithms by USGS practical rating
# ----------------------------------------------------
rank_metric = "USGS_Mean_Accuracy_Rating_mean"

rank_df = convergence_summary.copy()

rank_df = rank_df.sort_values(
    by=[
        "n_repetitions",
        "scenario",
        "severity",
        "parameter",
        rank_metric,
        "RMSE_mean",
        "MAE_mean"
    ],
    ascending=[True, True, True, True, True, True, True]
)

best_by_rep = (
    rank_df
    .groupby(
        ["n_repetitions", "scenario", "severity", "parameter"],
        as_index=False
    )
    .first()
)

best_by_rep = best_by_rep[
    [
        "n_repetitions",
        "scenario",
        "severity",
        "parameter",
        "algorithm",
        "USGS_Mean_Accuracy_Rating_mean",
        "USGS_Good_or_Better_Rate_mean",
        "RMSE_mean",
        "MAE_mean"
    ]
].copy()

best_by_rep.to_csv(
    "best_algorithm_by_repetition_level_10_20_30.csv",
    index=False
)

# ----------------------------------------------------
# Compare 10 and 20 to 30
# ----------------------------------------------------
best_30 = best_by_rep[
    best_by_rep["n_repetitions"] == 30
][
    ["scenario", "severity", "parameter", "algorithm"]
].rename(
    columns={"algorithm": "algorithm_ref30"}
)

best_compare = best_by_rep[
    best_by_rep["n_repetitions"] < 30
].merge(
    best_30,
    on=["scenario", "severity", "parameter"],
    how="left"
)

best_compare["same_as_30"] = (
    best_compare["algorithm"] == best_compare["algorithm_ref30"]
)

rank_stability = (
    best_compare
    .groupby("n_repetitions", as_index=False)
    .agg(
        same_best_algorithm_rate=("same_as_30", "mean")
    )
)

rank_stability["same_best_algorithm_rate_percent"] = (
    100 * rank_stability["same_best_algorithm_rate"]
)

rank_stability.to_csv(
    "rank_stability_against_30reps.csv",
    index=False
)

rank_stability


## Hydrologic-phase figures


In [ ]:
# Source development-notebook cell 147
# ============================================================
# SPLIT PARITY FIGURE INTO TWO 4 × 4 FIGURES
#
# Figure A:
#   Linear, PCHIP, Spline, Moving mean
#
# Figure B:
#   ΔH weighted, HPWI, Kalman, SSA gap
# ============================================================

figure_groups = {

    "A": [
        "Linear_Interpolation",
        "PCHIP",
        "Cubic_Spline",
        "Local_Moving_Mean"
    ],

    "B": [
        "Delta_H_Weighted_Interpolation",
        "HPWI",
        "Kalman_Filter",
        "SSA_Gap_Filling"
    ]
}


# ============================================================
# COMMON ROBUST AXIS LIMITS BY PARAMETER
#
# Uses ALL 8 algorithms so Figure A and Figure B have
# identical scales for each water-quality parameter.
#
# Extreme 0.2% tails are excluded from the DISPLAY ONLY.
# All observations remain in MAE, RMSE, r², etc.
# ============================================================

parameter_limits = {}

for parameter in parameter_order:

    parameter_data = pointwise[
        pointwise["parameter"] == parameter
    ].copy()

    if parameter_data.empty:
        raise ValueError(
            f"No pointwise data found for parameter: {parameter}"
        )

    all_values = np.concatenate([
        parameter_data["reference"].to_numpy(dtype=float),
        parameter_data["reconstructed"].to_numpy(dtype=float)
    ])

    all_values = all_values[
        np.isfinite(all_values)
    ]

    if len(all_values) == 0:
        raise ValueError(
            f"No finite plotting values found for parameter: {parameter}"
        )

    # --------------------------------------------------------
    # Robust limits, consistent with original figure
    # --------------------------------------------------------

    lo = np.nanpercentile(
        all_values,
        0.2
    )

    hi = np.nanpercentile(
        all_values,
        99.8
    )

    padding = (
        0.04 * (hi - lo)
        if hi > lo
        else 1.0
    )

    lo = lo - padding
    hi = hi + padding

    # --------------------------------------------------------
    # Turbidity cannot be physically negative.
    # This changes DISPLAY ONLY, not predictions or statistics.
    # --------------------------------------------------------

    if parameter == "Turbidity":
        lo = 0.0

    parameter_limits[parameter] = (
        lo,
        hi
    )


# ============================================================
# VERIFY PARAMETER LIMITS BEFORE PLOTTING
# ============================================================

print("/nParameter axis limits:")

for parameter in parameter_order:

    print(
        f"{parameter:10s}: "
        f"{parameter_limits[parameter][0]:.3f} "
        f"to "
        f"{parameter_limits[parameter][1]:.3f}"
    )

print(
    "/nKeys in parameter_limits:",
    list(parameter_limits.keys())
)

# ============================================================
# SHARED LEGEND HANDLES
# ============================================================

legend_handles = [

    Line2D(
        [0],
        [0],
        marker="o",
        linestyle="",
        markersize=7,
        markerfacecolor=phase_colors[phase],
        markeredgecolor="none",
        label=phase
    )

    for phase in phase_order
]


legend_handles += [

    Line2D(
        [0],
        [0],
        linestyle="--",
        color="black",
        linewidth=1.1,
        label="1:1 line"
    ),

    Line2D(
        [0],
        [0],
        linestyle="-",
        color="gray",
        linewidth=1.1,
        label="Linear fit"
    )
]


# ============================================================
# CREATE EACH 4 × 4 FIGURE
# ============================================================

saved_figures = []


for group_name, group_algorithms in figure_groups.items():

    fig, axes = plt.subplots(
        nrows=4,
        ncols=4,
        figsize=(13.5, 12),
        facecolor="white"
    )


    # ========================================================
    # PARAMETER ROWS
    # ========================================================

    for row, parameter in enumerate(parameter_order):

        parameter_data = pointwise[
            pointwise["parameter"] == parameter
        ]

        lo, hi = parameter_limits[
            parameter
        ]


        # ====================================================
        # ALGORITHM COLUMNS
        # ====================================================

        for col, algorithm in enumerate(group_algorithms):

            ax = axes[
                row,
                col
            ]

            data = parameter_data[
                parameter_data["algorithm"] == algorithm
            ]


            # =================================================
            # SCATTER BY HYDROLOGIC PHASE
            # =================================================

            for phase_index, phase in enumerate(phase_order):

                d = data[
                    data["hydrologic_phase"] == phase
                ]

                if d.empty:
                    continue


                # Visual subsampling only
                d_plot = subsample_for_plot(
                    d,
                    max_n=MAX_POINTS_PER_PHASE,
                    seed=(
                        1000
                        + row * 100
                        + col * 10
                        + phase_index
                    )
                )


                ax.scatter(
                    d_plot["reference"],
                    d_plot["reconstructed"],
                    s=8,
                    alpha=0.25,
                    color=phase_colors[phase],
                    edgecolors="none",
                    rasterized=True,
                    zorder=2
                )


            # =================================================
            # 1:1 AGREEMENT LINE
            # =================================================

            ax.plot(
                [lo, hi],
                [lo, hi],
                "--",
                color="black",
                linewidth=1.1,
                zorder=5
            )


            # =================================================
            # LINEAR FIT + STATISTICS
            #
            # Statistics use ALL points, not subsampled points.
            # =================================================

            valid = (
                data[
                    [
                        "reference",
                        "reconstructed"
                    ]
                ]
                .dropna()
            )


            if len(valid) > 2:

                x = valid[
                    "reference"
                ].to_numpy()

                y = valid[
                    "reconstructed"
                ].to_numpy()


                slope, intercept = np.polyfit(
                    x,
                    y,
                    1
                )


                xx = np.linspace(
                    lo,
                    hi,
                    100
                )

                yy = (
                    slope * xx
                    + intercept
                )


                ax.plot(
                    xx,
                    yy,
                    color="gray",
                    linewidth=1.1,
                    zorder=4
                )


                r = np.corrcoef(
                    x,
                    y
                )[0, 1]

                r2 = r ** 2


                rmse = np.sqrt(
                    np.mean(
                        (y - x) ** 2
                    )
                )


                mae = np.mean(
                    np.abs(
                        y - x
                    )
                )


                ax.text(
                    0.04,
                    0.96,
                    (
                        f"$R^2$={r2:.2f}/n"
                        f"RMSE={rmse:.2f}/n"
                        f"MAE={mae:.2f}"
                    ),
                    transform=ax.transAxes,
                    va="top",
                    fontsize=8
                )


            # =================================================
            # COMMON AXES
            # =================================================

            ax.set_xlim(
                lo,
                hi
            )

            ax.set_ylim(
                lo,
                hi
            )


            # =================================================
            # ALGORITHM TITLE
            # =================================================

            if row == 0:

                ax.set_title(
                    algo_display.get(
                        algorithm,
                        algorithm
                    ),
                    fontsize=12,
                    fontweight="bold",
                    pad=7
                )


            # =================================================
            # Y AXIS
            # =================================================

            if col == 0:

                ax.set_ylabel(
                    (
                        f"{parameter_display.get(parameter, parameter)}/n"
                        "Reconstructed value"
                    ),
                    fontsize=10
                )


            # =================================================
            # X AXIS
            # =================================================

            if row == len(parameter_order) - 1:

                ax.set_xlabel(
                    "Reference value",
                    fontsize=10
                )


            ax.tick_params(
                labelsize=8
            )


    # ========================================================
    # SHARED LEGEND FOR EACH FIGURE
    # ========================================================

    fig.legend(
        handles=legend_handles,
        loc="lower center",
        ncol=5,
        frameon=False,
        fontsize=10,
        bbox_to_anchor=(0.5, 0.012)
    )


    # ========================================================
    # SPACING
    # ========================================================

    fig.subplots_adjust(
        left=0.09,
        right=0.99,
        top=0.95,
        bottom=0.095,
        wspace=0.22,
        hspace=0.25
    )


    # ========================================================
    # SAVE
    # ========================================================

    png_path = (
        output_dir
        /
        f"{scenario}_pointwise_parity_hydrologic_phase_"
        f"8alg_part_{group_name}.png"
    )


    pdf_path = (
        output_dir
        /
        f"{scenario}_pointwise_parity_hydrologic_phase_"
        f"8alg_part_{group_name}.pdf"
    )


    fig.savefig(
        png_path,
        dpi=600,
        bbox_inches="tight",
        facecolor="white"
    )


    fig.savefig(
        pdf_path,
        bbox_inches="tight",
        facecolor="white"
    )


    saved_figures.append(
        (
            group_name,
            png_path,
            pdf_path
        )
    )


    plt.show()


# ============================================================
# FINAL SAVED-FIGURE REPORT
# ============================================================

print(
    "/n========================================"
)

print(
    "4 × 4 PARITY FIGURES SAVED"
)

print(
    "========================================"
)


for group_name, png_path, pdf_path in saved_figures:

    print(
        f"/nFigure {group_name}:"
    )

    print(
        "PNG:",
        png_path.resolve()
    )

    print(
        "PDF:",
        pdf_path.resolve()
    )


In [ ]:
# Source development-notebook cell 148
import pandas as pd

# ============================================================
# COMMON ORDERS
# ============================================================

parameter_order = ["Turbidity", "EC", "DO", "Temp"]

algorithm_order = [
    "Linear_Interpolation",
    "PCHIP",
    "Cubic_Spline",
    "Kalman_Filter",
    "SSA_Gap_Filling",
        # New gap-filling algorithms
    "Local_Moving_Mean",
    "Delta_H_Weighted_Interpolation",
    "HPWI"
]

parameter_display = {
    "Turbidity": "Turbidity",
    "EC": "EC",
    "DO": "DO",
    "Temp": "Temperature"
}

algorithm_display = {
    "Linear_Interpolation": "Linear",
    "PCHIP": "PCHIP",
    "Cubic_Spline": "Spline",
    "Kalman_Filter": "Kalman",
    "SSA_Gap_Filling": "SSA gap",
        # New gap-filling algorithms
    "Local_Moving_Mean": "Moving mean",
    "Delta_H_Weighted_Interpolation": "ΔH weighted",
    "HPWI": "HPWI",
}


# ============================================================
# TABLE S4 — SEVERITY-SPECIFIC RANKS
# ============================================================

rank = pd.read_csv(
    "gap_filling_RMSE_MAE_rank_summary_by_severity.csv"
)

rank = rank[
    rank["scenario"] == "random_missing"
].copy()

rank["severity"] = rank["severity"].str.lower()

# Mean rank only — cleaner for supplementary table
rmse = rank.pivot_table(
    index=["parameter", "algorithm"],
    columns="severity",
    values="Mean_RMSE_Rank"
)

mae = rank.pivot_table(
    index=["parameter", "algorithm"],
    columns="severity",
    values="Mean_MAE_Rank"
)

table_s4 = pd.DataFrame(index=rmse.index)

table_s4["Moderate RMSE Rank"] = rmse["moderate"]
table_s4["Moderate MAE Rank"] = mae["moderate"]
table_s4["Severe RMSE Rank"] = rmse["severe"]
table_s4["Severe MAE Rank"] = mae["severe"]

table_s4 = table_s4.reset_index()

# Display names
table_s4["Parameter"] = table_s4["parameter"].map(parameter_display)
table_s4["Algorithm"] = table_s4["algorithm"].map(algorithm_display)

# Sort properly
table_s4["parameter"] = pd.Categorical(
    table_s4["parameter"],
    categories=parameter_order,
    ordered=True
)

table_s4["algorithm"] = pd.Categorical(
    table_s4["algorithm"],
    categories=algorithm_order,
    ordered=True
)

table_s4 = table_s4.sort_values(
    ["parameter", "algorithm"]
)

# Round
rank_cols = [
    "Moderate RMSE Rank",
    "Moderate MAE Rank",
    "Severe RMSE Rank",
    "Severe MAE Rank"
]

table_s4[rank_cols] = table_s4[rank_cols].round(2)

# Final column order
table_s4 = table_s4[
    ["Parameter", "Algorithm"] + rank_cols
]

table_s4.to_csv(
    "Table_S4_random_missingness_severity_ranks_CLEAN.csv",
    index=False
)

print("/nTABLE S4")
print(table_s4.to_string(index=False))


# ============================================================
# TABLE S5 — HYDROLOGIC-PHASE PERFORMANCE
# ============================================================

phase = pd.read_csv(
    "supplementary_hydrologic_phase_analysis/"
    "performance_summary_by_hydrologic_phase.csv"
)

# Normalize phase labels just in case
phase["Hydrologic_Phase"] = phase["Hydrologic_Phase"].replace({
    "Baseline": "Baseline/inter-event"
})

mae = phase.pivot_table(
    index=["parameter", "algorithm"],
    columns="Hydrologic_Phase",
    values="Mean_MAE"
)

rmse = phase.pivot_table(
    index=["parameter", "algorithm"],
    columns="Hydrologic_Phase",
    values="Mean_RMSE"
)

table_s5 = pd.DataFrame(index=mae.index)

table_s5["Baseline MAE"] = mae["Baseline/inter-event"]
table_s5["Baseline RMSE"] = rmse["Baseline/inter-event"]

table_s5["Storm-limb MAE"] = mae["Storm limb"]
table_s5["Storm-limb RMSE"] = rmse["Storm limb"]

table_s5["Peak-window MAE"] = mae["Peak window"]
table_s5["Peak-window RMSE"] = rmse["Peak window"]

table_s5 = table_s5.reset_index()

# Display names
table_s5["Parameter"] = table_s5["parameter"].map(parameter_display)
table_s5["Algorithm"] = table_s5["algorithm"].map(algorithm_display)

# Sort
table_s5["parameter"] = pd.Categorical(
    table_s5["parameter"],
    categories=parameter_order,
    ordered=True
)

table_s5["algorithm"] = pd.Categorical(
    table_s5["algorithm"],
    categories=algorithm_order,
    ordered=True
)

table_s5 = table_s5.sort_values(
    ["parameter", "algorithm"]
)

metric_cols = [
    "Baseline MAE",
    "Baseline RMSE",
    "Storm-limb MAE",
    "Storm-limb RMSE",
    "Peak-window MAE",
    "Peak-window RMSE"
]

table_s5[metric_cols] = table_s5[metric_cols].round(3)

table_s5 = table_s5[
    ["Parameter", "Algorithm"] + metric_cols
]

table_s5.to_csv(
    "Table_S5_random_missingness_hydrologic_phase_CLEAN.csv",
    index=False
)

print("/nTABLE S5")
print(table_s5.to_string(index=False))
